# 05b — Jev prompt optimization

이 노트북은 모델을 학습하거나 교체하지 않고, Jev 1.13의 질문 instructions만 v4와 v5 사이에서 짝지어 비교합니다. 두 조건 모두 03에서 정한 20개 특성에 05a와 같은 train-only Robust 변환을 적용하고 배율은 1×로 고정합니다.

요청은 OpenRouter Decisions API의 typesafe/jev-1.13에만 보냅니다. API에는 모델, Robust 변환된 숫자 특성, decision question만 보냅니다. 타깃·TTF·구간·그룹·파일·샘플 식별자는 로컬 평가에만 남깁니다.

실행 순서: 05a/스케일 결과 확인 → validation 표본·그룹 감사 → 프롬프트/요청 감사 → 90회 파일럿 → 개발 비교 및 선택 고정 → holdout 확인 → 반복성·비용·결과 저장. 파일럿 실패 시 개발/holdout 호출은 생략합니다.
## Offline follow-up
The original 05b API run and its saved outputs remain an archive. The default notebook path executes only the offline diagnostics at the end and makes no network requests.


In [ ]:
# The completed 552-call API experiment is archived below. Offline diagnostics are the default.
ARCHIVED_API_EXECUTION = False


In [ ]:
if ARCHIVED_API_EXECUTION:
    # Setup, secret loading, and historical-result verification.
    from pathlib import Path
    import os, sys, json, math, hashlib, time, uuid, urllib.request, urllib.error
    ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                 if (p / "artifacts/jev_tut/config.json").is_file()), None)
    if ROOT is None:
        raise FileNotFoundError("Project root or 05a artifacts not found.")
    venv_site = ROOT / ".venv" / "Lib" / "site-packages"
    if venv_site.is_dir():
        sys.path.insert(0, str(venv_site))
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    from sklearn.model_selection import GroupShuffleSplit
    from sklearn.metrics import roc_auc_score, average_precision_score

    OUT = ROOT / "artifacts" / "jev_prompt_optimization"
    CACHE_DIR = OUT / "cache"
    OUT.mkdir(parents=True, exist_ok=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    RUN_ID = str(uuid.uuid4())
    SEED = 42
    MODEL_ID = "typesafe/jev-1.13"
    DECISIONS_URL = "https://openrouter.ai/api/alpha/decisions"
    DOCS_URL = "https://openrouter.ai/docs/api/api-reference/alphadecisions/submit-a-decisions-request"
    MAX_API_CALLS = 552
    HARD_COST_CAP_USD = 0.05
    PROMPT_IDS = ("v4", "v5")
    REGIONS = ("normal", "boundary", "near_failure")

    def load_env_keys(path):
        if not path.is_file():
            return
        for line in path.read_text(encoding="utf-8").splitlines():
            text = line.strip()
            if text and not text.startswith("#") and "=" in text:
                key, value = text.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                if key in {"JEV_API_KEY", "OPENROUTER_API_KEY"} and value:
                    os.environ.setdefault(key, value)

    load_env_keys(ROOT / ".env")
    API_KEY = os.environ.get("JEV_API_KEY") or os.environ.get("OPENROUTER_API_KEY")
    if not API_KEY:
        raise RuntimeError("Set JEV_API_KEY or OPENROUTER_API_KEY in the environment or project .env.")

    # Recompute the requested historical facts directly from saved artifacts.
    scale_cache_paths = sorted((ROOT / "artifacts/jev_scale_sensitivity/cache").glob("*.json"))
    scale_cache = [json.loads(p.read_text(encoding="utf-8")) for p in scale_cache_paths]
    scale_inputs = [int(r["usage"]["input_tokens"]) for r in scale_cache if r.get("usage") and r["usage"].get("input_tokens") is not None]
    scale_outputs = [int(r["usage"]["output_tokens"]) for r in scale_cache if r.get("usage") and r["usage"].get("output_tokens") is not None]
    scale_costs = [float(r["usage"]["cost"]) for r in scale_cache if r.get("usage") and r["usage"].get("cost") is not None]
    scale_model_counts = pd.Series([r.get("model") for r in scale_cache]).value_counts().to_dict()
    scale_provider_counts = pd.Series([r.get("provider") for r in scale_cache]).value_counts().to_dict()
    scale_parse_errors = sum(not bool(r.get("parse_success")) for r in scale_cache)
    if len(scale_cache) != 141 or sum(scale_inputs) != 138828 or sum(scale_outputs) != 2538:
        raise ValueError("Saved scale-sensitivity response/usage totals differ from the requested audit values.")
    if not np.isclose(sum(scale_costs), 0.005830776, atol=1e-12):
        raise ValueError("Saved scale-sensitivity cost differs from the requested audit value.")
    if scale_model_counts != {"typesafe/jev-1.13-20260917": 141} or scale_provider_counts != {"TypeSafe": 141} or scale_parse_errors:
        raise ValueError("Historical scale-sensitivity model/provider/parsing audit failed.")

    old_robust = pd.read_csv(ROOT / "artifacts/jev_tut/robust_results.csv", low_memory=False)
    old_robust = old_robust.loc[old_robust.parse_success.fillna(False)].copy()
    old_region_means = old_robust.groupby(["target", "region"]).risk_score.mean().unstack()
    old_separation = float((old_region_means["near_failure"] - old_region_means["normal"]).mean())
    old_ordered = int(((old_region_means["normal"] < old_region_means["boundary"]) &
                       (old_region_means["boundary"] < old_region_means["near_failure"])).sum())
    scale_macro = pd.read_csv(ROOT / "artifacts/jev_scale_sensitivity/macro_comparison.csv")
    scale_control = scale_macro.loc[scale_macro.representation == "control_1p0"].iloc[0]
    scale_down = scale_macro.loc[scale_macro.representation == "down_0p5"].iloc[0]
    scale_up = scale_macro.loc[scale_macro.representation == "up_2p0"].iloc[0]
    if round(old_separation, 3) != 0.234 or old_ordered != 3:
        raise ValueError("05a Robust baseline does not match 0.234 / 3-of-3.")
    if (round(float(scale_control.separation), 3) != 0.249 or
        round(float(scale_control.ROC_AUC), 3) != 0.973 or
        round(float(scale_control.PR_AUC), 3) != 0.981 or
        int(scale_control.ordered_targets) != 3):
        raise ValueError("Scale 1x control does not match the recorded reference.")
    if (float(scale_down.separation) > float(scale_control.separation) or
        float(scale_up.separation) > float(scale_control.separation)):
        raise ValueError("Scale-sensitivity assumption changed: a non-1x factor improved separation.")

    HISTORICAL_MEAN_COST = float(np.mean(scale_costs))
    HISTORICAL_COST_RESERVE = float(np.max(scale_costs))
    planned_calls = 270 * 2 + 6 * 2
    estimated_input_tokens = planned_calls * float(np.mean(scale_inputs))
    estimated_output_tokens = planned_calls * float(np.mean(scale_outputs))
    estimated_cost = planned_calls * HISTORICAL_MEAN_COST
    hist_summary = pd.DataFrame([
        {"check": "05a Robust", "separation": old_separation, "ROC_AUC": np.nan, "PR_AUC": np.nan, "ordered_targets": old_ordered},
        {"check": "scale 1x", "separation": float(scale_control.separation), "ROC_AUC": float(scale_control.ROC_AUC),
         "PR_AUC": float(scale_control.PR_AUC), "ordered_targets": int(scale_control.ordered_targets)},
        {"check": "scale 0.5x", "separation": float(scale_down.separation), "ROC_AUC": float(scale_down.ROC_AUC),
         "PR_AUC": float(scale_down.PR_AUC), "ordered_targets": int(scale_down.ordered_targets)},
        {"check": "scale 2x", "separation": float(scale_up.separation), "ROC_AUC": float(scale_up.ROC_AUC),
         "PR_AUC": float(scale_up.PR_AUC), "ordered_targets": int(scale_up.ordered_targets)},
    ])
    display(hist_summary.round(3))
    print("Historical scale responses:", len(scale_cache), "| model/provider:", scale_model_counts, scale_provider_counts,
          "| parse errors:", scale_parse_errors)
    print("Historical usage: input", sum(scale_inputs), "| mean", round(float(np.mean(scale_inputs)), 1),
          "| p95", int(np.quantile(scale_inputs, 0.95, method="higher")),
          "| output", sum(scale_outputs), "| mean", round(float(np.mean(scale_outputs)), 1),
          "| reported cost USD", f"{sum(scale_costs):.9f}")
    print("Plan for 552 calls: input", round(estimated_input_tokens), "| output", round(estimated_output_tokens),
          "| estimated reported cost USD", f"{estimated_cost:.4f}", "| hard caps", MAX_API_CALLS, f"${HARD_COST_CAP_USD:.2f}")
    print("Pinned model:", MODEL_ID, "| endpoint:", DECISIONS_URL, "| credential configured (value hidden):", bool(API_KEY))
    print("Artifacts:", OUT)

## 1. Sample design and group audit

The analysis unit is the processed 100-observation window from notebook 02. The group key is the saved 03 definition, source_file plus sequence_index. Validation remains the fixed 03 validation group set; a seeded group split assigns 25% of those groups to this prompt holdout. No group can appear in both prompt-development and prompt-holdout samples.

The 20 features and target columns are read from saved 03/05a artifacts. Regions use the recorded TTF cutoffs: normal > 50,000 seconds; boundary > 5,000 and <= 50,000; near_failure <= 5,000. The 45 05a rows are excluded by target/file/sequence/window. This cell prints availability and selected counts before any API request.

In [ ]:
if ARCHIVED_API_EXECUTION:
    # Build a bounded reservoir sample while counting every eligible validation row.
    base_config = json.loads((ROOT / "artifacts/jev_tut/config.json").read_text(encoding="utf-8"))
    split_meta = json.loads((ROOT / "artifacts/random_forest/ttf_baseline_split_metadata.json").read_text(encoding="utf-8"))
    features = list(base_config["feature_columns"])
    targets = list(base_config["target_columns"])
    target_labels = dict(split_meta["target_labels"])
    if base_config["model_id"] != MODEL_ID or len(features) != 20 or len(set(features)) != 20:
        raise ValueError("05a model or compact feature audit failed.")
    if base_config["split_seed"] != SEED or split_meta["group_overlap"] != 0:
        raise ValueError("Fixed 03 group split metadata failed audit.")
    if split_meta["group_definition"] != "source_file + '::' + sequence_index":
        raise ValueError("Unexpected saved analysis group definition.")

    split_path = ROOT / "artifacts/random_forest/ttf_baseline_group_split.csv"
    split_table = pd.read_csv(split_path, usecols=["group_id", "split"], dtype={"group_id": str, "split": str})
    validation_group_ids = sorted(split_table.loc[split_table.split == "validation", "group_id"].astype(str).unique())
    if len(validation_group_ids) != int(split_meta["validation_groups"]):
        raise ValueError("Validation group count differs from saved metadata.")
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
    _, holdout_indices = next(gss.split(np.zeros(len(validation_group_ids)), groups=validation_group_ids))
    HOLDOUT_GROUPS = {validation_group_ids[i] for i in holdout_indices}
    DEVELOPMENT_GROUPS = set(validation_group_ids) - HOLDOUT_GROUPS
    if DEVELOPMENT_GROUPS & HOLDOUT_GROUPS:
        raise AssertionError("Development and holdout group IDs overlap.")

    old_samples = pd.read_csv(ROOT / "artifacts/jev_tut/selected_samples.csv",
                              usecols=["target", "source_file", "sequence_index", "window_index", "sample_id"],
                              low_memory=False)
    def index_text(value):
        return str(int(float(value)))
    def local_row_key(target, source_file, sequence_index, window_index):
        return (str(target), str(source_file), index_text(sequence_index), int(float(window_index)))
    old_sample_keys = {local_row_key(r.target, r.source_file, r.sequence_index, r.window_index)
                       for r in old_samples.itertuples(index=False)}
    if len(old_samples) != 45 or len(old_sample_keys) != 45:
        raise ValueError("Expected 45 unique 05a sample rows for exclusion.")

    ROW_CAP = 30
    reservoirs, reservoir_rngs, available_groups = {}, {}, {}
    availability_counts = {}
    for t in targets:
        for region in REGIONS:
            for subset in ("development", "holdout"):
                key = (t, region, subset)
                reservoirs[key] = []
                digest = hashlib.sha256("|".join((str(SEED), *key)).encode("utf-8")).digest()
                reservoir_rngs[key] = np.random.default_rng(int.from_bytes(digest[:8], "little"))
                available_groups[key] = set()
                availability_counts[key] = 0

    def region_codes(values):
        y = np.asarray(values, dtype=float)
        out = np.full(len(y), "", dtype=object)
        finite = np.isfinite(y)
        out[finite & (y > 50000)] = "normal"
        out[finite & (y > 5000) & (y <= 50000)] = "boundary"
        out[finite & (y <= 5000)] = "near_failure"
        return out

    feature_dir = ROOT / "data/interim/phm2018_jev/features/train"
    feature_paths = sorted(feature_dir.glob("*.csv"))
    if not feature_paths:
        raise FileNotFoundError(f"No processed train feature CSVs found under {feature_dir}")
    usecols = ["source_file", "sequence_index", "window_index", *features, *targets]
    for path in feature_paths:
        header = pd.read_csv(path, nrows=0).columns.tolist()
        missing = sorted(set(usecols) - set(header))
        if missing:
            raise ValueError(f"{path.name} lacks columns: {missing}")

    for file_i, path in enumerate(feature_paths, 1):
        print(f"Scanning eligible rows {file_i}/{len(feature_paths)}: {path.name}", flush=True)
        for chunk in pd.read_csv(path, usecols=usecols, chunksize=100000, low_memory=False):
            seq = pd.to_numeric(chunk.sequence_index, errors="coerce")
            seq_text = seq.astype("Int64").astype(str)
            group_ids = chunk.source_file.astype(str) + "::" + seq_text
            valid = group_ids.isin(validation_group_ids).to_numpy(copy=True)
            if not valid.any():
                continue
            x = chunk[features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
            complete = np.isfinite(x).all(axis=1)
            valid &= complete
            if not valid.any():
                continue
            source_values = chunk.source_file.astype(str).to_numpy()
            seq_values = seq.to_numpy(dtype=float)
            win_values = pd.to_numeric(chunk.window_index, errors="coerce").to_numpy(dtype=float)
            group_values = group_ids.to_numpy(dtype=str)
            subset_values = np.where(np.isin(group_values, list(HOLDOUT_GROUPS)), "holdout", "development")
            for target in targets:
                y = pd.to_numeric(chunk[target], errors="coerce").to_numpy(dtype=float)
                regions = region_codes(y)
                eligible_positions = np.flatnonzero(valid & (regions != ""))
                for pos in eligible_positions:
                    target_key = local_row_key(target, source_values[pos], seq_values[pos], win_values[pos])
                    if target_key in old_sample_keys:
                        continue
                    region = str(regions[pos])
                    subset = str(subset_values[pos])
                    key = (target, region, subset)
                    availability_counts[key] += 1
                    group_id = str(group_values[pos])
                    available_groups[key].add(group_id)
                    slug = target.removeprefix("ttf_").removesuffix("_seconds")
                    sample_id = f"{slug}|{source_values[pos]}|{index_text(seq_values[pos])}|w{int(win_values[pos])}"
                    record = {
                        "sample_id": sample_id, "target": target, "target_label": target_labels[target],
                        "region": region, "subset": subset, "source_file": source_values[pos],
                        "group_id": group_id, "sequence_index": int(seq_values[pos]),
                        "window_index": int(win_values[pos]), "ttf_seconds": float(y[pos]),
                    }
                    record.update({f: float(x[pos, j]) for j, f in enumerate(features)})
                    reservoir = reservoirs[key]
                    seen = availability_counts[key]
                    if len(reservoir) < ROW_CAP:
                        reservoir.append(record)
                    else:
                        replacement = int(reservoir_rngs[key].integers(0, seen))
                        if replacement < ROW_CAP:
                            reservoir[replacement] = record

    availability_rows = []
    for target in targets:
        for region in REGIONS:
            dev_key, hold_key = (target, region, "development"), (target, region, "holdout")
            dev_n, hold_n = availability_counts[dev_key], availability_counts[hold_key]
            availability_rows.append({
                "target": target, "target_label": target_labels[target], "region": region,
                "eligible_after_05a_exclusion": dev_n + hold_n,
                "development_available": dev_n, "development_groups_available": len(available_groups[dev_key]),
                "holdout_available": hold_n, "holdout_groups_available": len(available_groups[hold_key]),
            })
    availability_audit = pd.DataFrame(availability_rows)
    # Keep the requested 2:1 development/holdout allocation; lower every cell uniformly only if needed.
    n_holdout_per_cell = min(10, min(availability_counts[(t, r, "holdout")] for t in targets for r in REGIONS),
                             min(availability_counts[(t, r, "development")] for t in targets for r in REGIONS) // 2)
    n_development_per_cell = 2 * n_holdout_per_cell
    if n_holdout_per_cell < 1:
        raise ValueError("At least one shared holdout sample per cell is unavailable after group splitting.")
    print("API has not been called. Eligible counts after excluding the 45 prior 05a rows:")
    display(availability_audit)
    print(f"Common balanced allocation: {n_development_per_cell} development + {n_holdout_per_cell} holdout per target-region cell.")

    selected_records = []
    for target in targets:
        for region in REGIONS:
            for subset, count in (("development", n_development_per_cell), ("holdout", n_holdout_per_cell)):
                reservoir = reservoirs[(target, region, subset)]
                if len(reservoir) < count:
                    raise ValueError(f"Reservoir underflow for {target}/{region}/{subset}.")
                selected_records.extend(reservoir[:count])
    selected_samples = pd.DataFrame(selected_records).sort_values(
        ["subset", "target", "region", "sample_id"], kind="stable").reset_index(drop=True)
    if selected_samples.duplicated(["target", "source_file", "sequence_index", "window_index"]).any():
        raise AssertionError("Duplicate selected row within target.")
    dev_groups = set(selected_samples.loc[selected_samples.subset == "development", "group_id"])
    hold_groups = set(selected_samples.loc[selected_samples.subset == "holdout", "group_id"])
    if dev_groups & hold_groups:
        raise AssertionError("A sampled group occurs in both development and holdout.")

    selected_counts = selected_samples.groupby(["subset", "target", "region"]).size().rename("selected").reset_index()
    display(selected_counts)
    print("Selected rows:", len(selected_samples), "| development:", int((selected_samples.subset == "development").sum()),
          "| holdout:", int((selected_samples.subset == "holdout").sum()),
          "| sampled groups:", selected_samples.group_id.nunique(),
          "| sampled-group overlap:", len(dev_groups & hold_groups),
          "| prior 05a sample rows excluded:", len(old_sample_keys))

    sample_metadata_columns = ["sample_id", "target", "target_label", "region", "subset",
                               "source_file", "group_id", "sequence_index", "window_index", "ttf_seconds"]
    sample_audit = selected_samples[sample_metadata_columns].copy()
    availability_audit.to_csv(OUT / "sample_availability_audit.csv", index=False)
    sample_audit.to_csv(OUT / "sample_audit.csv", index=False)
    if len(selected_samples) != 270:
        print("The common cell size was reduced because availability was below the target; no rows were duplicated.")

## 2. Paired prompts and request-body audit

v4 is loaded from the saved 05a prompt file without editing its question. v5 keeps the same score type and three criteria, and changes only instructions. Both prompt variants receive identical 1× Robust-transformed feature values for each sample.

The request-body audit checks the exact top-level fields and nested state fields before any network request. Feature values are not printed or written to artifacts; labels stay local for evaluation.

In [ ]:
if ARCHIVED_API_EXECUTION:
    # Exact v4 question and the requested v5 instructions.
    prompt_text_v4 = (ROOT / "artifacts/jev_tut/prompt_v4.txt").read_text(encoding="utf-8")
    QUESTION_V4 = json.loads(prompt_text_v4.split("question=\n", 1)[1])
    V5_INSTRUCTIONS = (
        "Using only the numeric measurements in state.features, assess the equipment’s current condition, not time-to-failure. "
        "Consider the overall pattern and strength of evidence across the features. Use Normal when evidence of abnormality is weak, "
        "Uncertain when evidence is mixed or insufficient, and Degrading when the measurements support an abnormal condition. "
        "Do not decide from one value alone without considering the remaining measurements."
    )
    QUESTION_V5 = {"type": QUESTION_V4["type"], "instructions": V5_INSTRUCTIONS, "criteria": list(QUESTION_V4["criteria"])}
    QUESTIONS = {"v4": QUESTION_V4, "v5": QUESTION_V5}
    PROMPT_VERSIONS = {"v4": "jev_prompt_v4", "v5": "jev_prompt_v5"}
    if QUESTION_V4["type"] != "score" or len(QUESTION_V4["criteria"]) != 3:
        raise ValueError("Saved v4 structure is not the expected three-level score question.")
    if QUESTION_V4["criteria"] != QUESTION_V5["criteria"] or QUESTION_V4["type"] != QUESTION_V5["type"]:
        raise AssertionError("v4/v5 type and criteria must remain identical.")
    v5_lower = V5_INSTRUCTIONS.casefold()
    if any(feature.casefold() in v5_lower for feature in features):
        raise AssertionError("v5 instructions contain a selected feature name.")
    if any(t.casefold() in v5_lower for t in targets) or any(label.casefold() in v5_lower for label in target_labels.values()):
        raise AssertionError("v5 instructions contain a target name or fault label.")
    if any(token in v5_lower for token in ("5000", "50000", "near_failure", "boundary")):
        raise AssertionError("v5 instructions contain a TTF band.")

    scaler_stats = json.loads((ROOT / "artifacts/jev_tut/train_scaler_stats.json").read_text(encoding="utf-8"))
    if scaler_stats.get("source") != "03 train groups only":
        raise ValueError("Robust statistics are not confirmed as train-only.")

    def robust_feature_values(sample):
        target = str(sample["target"])
        values = {}
        for feature in features:
            scale = scaler_stats["targets"][target][feature]
            value = (float(sample[feature]) - float(scale["median_approx"])) / float(scale["safe_iqr"])
            if not math.isfinite(value):
                raise ValueError(f"Non-finite transformed value for {feature}.")
            values[feature] = float(value)
        return values

    def make_payload(sample, prompt_id):
        values = robust_feature_values(sample)
        payload = {
            "model": MODEL_ID,
            "state": {"features": values},
            "questions": {"equipment_state": QUESTIONS[prompt_id]},
        }
        if set(payload) != {"model", "state", "questions"} or set(payload["state"]) != {"features"}:
            raise AssertionError("Unexpected request fields.")
        if set(payload["state"]["features"]) != set(features) or len(payload["state"]["features"]) != 20:
            raise AssertionError("Request must contain exactly the 20 selected numeric features.")
        if any(not isinstance(v, float) or not math.isfinite(v) for v in values.values()):
            raise AssertionError("All transmitted measurements must be finite numbers.")
        return payload

    def canonical_json(value):
        return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)
    def request_hash(payload):
        return hashlib.sha256(canonical_json(payload).encode("utf-8")).hexdigest()

    audit_sample = selected_samples.sort_values("sample_id").iloc[0]
    payload_v4_audit, payload_v5_audit = make_payload(audit_sample, "v4"), make_payload(audit_sample, "v5")
    assert payload_v4_audit["model"] == payload_v5_audit["model"]
    assert payload_v4_audit["state"]["features"] == payload_v5_audit["state"]["features"]
    assert payload_v4_audit["questions"]["equipment_state"]["criteria"] == payload_v5_audit["questions"]["equipment_state"]["criteria"]
    assert payload_v4_audit["questions"]["equipment_state"]["instructions"] != payload_v5_audit["questions"]["equipment_state"]["instructions"]
    request_body_audit = pd.DataFrame([
        {"prompt_id": "v4", "prompt_version": PROMPT_VERSIONS["v4"], "top_level_fields": ["model", "state", "questions"],
         "state_fields": ["features"], "feature_count": len(payload_v4_audit["state"]["features"]),
         "request_sha256": request_hash(payload_v4_audit)},
        {"prompt_id": "v5", "prompt_version": PROMPT_VERSIONS["v5"], "top_level_fields": ["model", "state", "questions"],
         "state_fields": ["features"], "feature_count": len(payload_v5_audit["state"]["features"]),
         "request_sha256": request_hash(payload_v5_audit)},
    ])
    display(pd.DataFrame([
        {"field": "type", "v4": QUESTION_V4["type"], "v5": QUESTION_V5["type"], "changed": False},
        {"field": "instructions", "v4": QUESTION_V4["instructions"], "v5": QUESTION_V5["instructions"], "changed": True},
        {"field": "criteria", "v4": QUESTION_V4["criteria"], "v5": QUESTION_V5["criteria"], "changed": False},
    ]))
    display(request_body_audit)
    (OUT / "decision_questions.json").write_text(
        json.dumps({"v4": QUESTION_V4, "v5": QUESTION_V5}, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8")
    (OUT / "request_body_audit.json").write_text(
        json.dumps({"model": MODEL_ID, "endpoint": DECISIONS_URL, "feature_count": 20,
                    "top_level_fields": ["model", "state", "questions"], "state_fields": ["features"],
                    "local_identifiers_or_labels_in_payload": False,
                    "same_robust_features_across_prompts": True, "only_question_instructions_change": True,
                    "sample_request_hashes": request_body_audit.request_sha256.tolist()},
                   ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("No target, label, TTF, region, group, tool, filename, or sample ID is included in the request body.")

## 3. Safe request cache and pilot gate

Successful, parsed responses are cached using local sample ID, prompt version, pinned model, request hash, and repeat tag. API errors, model/provider mismatches, missing usage fields, and parse failures are recorded in the attempt ledger but never reused as successful cache entries.

The pilot uses five development samples per target-region cell (45 sample rows × two prompts = 90 requests). A model/provider mismatch stops requests immediately. Any failed parsing or usage check blocks the full run.

In [ ]:
if ARCHIVED_API_EXECUTION:
    # OpenRouter Decisions API client; credentials are used only in the Authorization header.
    ATTEMPT_LEDGER_PATH = OUT / "api_attempts.jsonl"
    def read_jsonl(path):
        if not path.is_file():
            return []
        rows = []
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                rows.append(json.loads(line))
        return rows

    API_ATTEMPTS = read_jsonl(ATTEMPT_LEDGER_PATH)
    RUN_ATTEMPT_START = len(API_ATTEMPTS)
    API_CALLS_THIS_RUN = 0
    CACHE_HITS_THIS_RUN = 0
    SUCCESSFUL_CALLS_THIS_RUN = 0
    API_ERRORS_THIS_RUN = 0
    MODEL_MISMATCH = False
    BUDGET_STOP = False
    LAST_REQUEST_START = 0.0
    MIN_REQUEST_INTERVAL_SECONDS = 0.25
    EXPECTED_API_KEY_NAMES = ("JEV_API_KEY", "OPENROUTER_API_KEY")

    def append_jsonl(path, record):
        with path.open("a", encoding="utf-8", newline="\n") as f:
            f.write(json.dumps(record, ensure_ascii=False, allow_nan=False, default=str) + "\n")

    def ledger_cost(records=None):
        records = API_ATTEMPTS if records is None else records
        return float(sum(float(r.get("usage_cost_usd") or 0.0) for r in records))

    def cache_path(sample_id, prompt_id, payload_hash, repeat_tag):
        identity = {
            "sample_id": str(sample_id), "prompt_version": PROMPT_VERSIONS[prompt_id],
            "model": MODEL_ID, "request_hash": payload_hash, "repeat_tag": str(repeat_tag),
        }
        key = hashlib.sha256(canonical_json(identity).encode("utf-8")).hexdigest()
        return CACHE_DIR / f"{key}.json"

    def parsed_score(raw):
        answer = (raw.get("answers") or {}).get("equipment_state") or {}
        if answer.get("type") != "score":
            raise ValueError("Unexpected answer type.")
        score = float(answer["score"])
        confidence = float(answer["confidence"])
        probs = {str(i): float(answer["probabilities"][str(i)]) for i in range(3)}
        if not math.isfinite(score) or not 0 <= score <= 2:
            raise ValueError("Native score outside [0,2].")
        if not math.isfinite(confidence) or not 0 <= confidence <= 1:
            raise ValueError("Confidence outside [0,1].")
        if any(not math.isfinite(p) or not 0 <= p <= 1 for p in probs.values()):
            raise ValueError("Invalid score-level probability.")
        if not 0.95 <= sum(probs.values()) <= 1.05:
            raise ValueError("Score-level probabilities do not sum to one.")
        state = ("normal", "uncertain", "degrading")[max(range(3), key=lambda i: probs[str(i)])]
        return {"risk_score": score / 2.0, "native_score": score, "confidence": confidence,
                "predicted_state": state, "probabilities": probs}

    def cache_read(path, sample_id, prompt_id, payload_hash, repeat_tag):
        global CACHE_HITS_THIS_RUN
        if not path.is_file():
            return None
        try:
            saved = json.loads(path.read_text(encoding="utf-8"))
            valid = (saved.get("success") is True and saved.get("parse_success") is True and
                     saved.get("usage_valid") is True and saved.get("model") and
                     str(saved.get("model")).startswith(MODEL_ID) and saved.get("provider") == "TypeSafe" and
                     saved.get("sample_id") == str(sample_id) and
                     saved.get("prompt_version") == PROMPT_VERSIONS[prompt_id] and
                     saved.get("request_hash") == payload_hash and
                     saved.get("repeat_tag", "") == str(repeat_tag))
            if valid:
                CACHE_HITS_THIS_RUN += 1
                return {**saved, "cache_hit": True, "api_call": False}
        except (OSError, ValueError, TypeError):
            return None
        return None

    def invoke_jev(sample, prompt_id, subset, repeat_tag=""):
        global API_CALLS_THIS_RUN, SUCCESSFUL_CALLS_THIS_RUN, API_ERRORS_THIS_RUN
        global MODEL_MISMATCH, BUDGET_STOP, LAST_REQUEST_START
        payload = make_payload(sample, prompt_id)
        body_hash = request_hash(payload)
        sample_id = str(sample["sample_id"])
        cached = cache_read(cache_path(sample_id, prompt_id, body_hash, repeat_tag),
                            sample_id, prompt_id, body_hash, repeat_tag)
        if cached is not None:
            return {**cached, "subset": str(subset)}
        now = time.perf_counter()
        if LAST_REQUEST_START:
            time.sleep(max(0.0, MIN_REQUEST_INTERVAL_SECONDS - (now - LAST_REQUEST_START)))
        current_actual_cost = ledger_cost()
        calls_remaining_if_sent = MAX_API_CALLS - len(API_ATTEMPTS)
        projected_cost = current_actual_cost + HISTORICAL_COST_RESERVE * max(calls_remaining_if_sent, 0)
        if calls_remaining_if_sent <= 0:
            BUDGET_STOP = True
            return {"sample_id": sample_id, "prompt_id": prompt_id, "prompt_version": PROMPT_VERSIONS[prompt_id],
                    "subset": subset, "repeat_tag": repeat_tag, "request_hash": body_hash, "api_call": False,
                    "cache_hit": False, "success": False, "parse_success": False, "usage_valid": False,
                    "error": "Maximum API-call count reached.", "raw_response": None}
        if projected_cost > HARD_COST_CAP_USD:
            BUDGET_STOP = True
            return {"sample_id": sample_id, "prompt_id": prompt_id, "prompt_version": PROMPT_VERSIONS[prompt_id],
                    "subset": subset, "repeat_tag": repeat_tag, "request_hash": body_hash, "api_call": False,
                    "cache_hit": False, "success": False, "parse_success": False, "usage_valid": False,
                    "error": f"Projected reported cost {projected_cost:.6f} exceeds the hard cap.", "raw_response": None}

        headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
        request = urllib.request.Request(
            DECISIONS_URL, data=canonical_json(payload).encode("utf-8"), headers=headers, method="POST")
        start = time.perf_counter()
        LAST_REQUEST_START = start
        API_CALLS_THIS_RUN += 1
        status_code, raw_response, error_text = None, None, None
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                status_code = int(response.status)
                raw_text = response.read().decode("utf-8")
            raw_response = json.loads(raw_text)
        except urllib.error.HTTPError as exc:
            status_code = int(exc.code)
            raw_text = exc.read().decode("utf-8", errors="replace")
            try:
                raw_response = json.loads(raw_text)
            except ValueError:
                raw_response = {"body_text": raw_text[:3000]}
            error_text = f"HTTP {status_code}"
        except Exception as exc:
            error_text = f"{type(exc).__name__}: {str(exc)[:500]}"
        latency = time.perf_counter() - start
        model = raw_response.get("model") if isinstance(raw_response, dict) else None
        provider = raw_response.get("provider") if isinstance(raw_response, dict) else None
        usage = raw_response.get("usage") if isinstance(raw_response, dict) else None
        usage = usage if isinstance(usage, dict) else {}
        try:
            usage_input = int(usage["input_tokens"])
            usage_output = int(usage["output_tokens"])
            usage_cost = float(usage["cost"])
            usage_valid = usage_input >= 0 and usage_output >= 0 and math.isfinite(usage_cost) and usage_cost >= 0
        except (KeyError, TypeError, ValueError, OverflowError):
            usage_input, usage_output, usage_cost, usage_valid = None, None, None, False
        model_ok = isinstance(model, str) and model.startswith(MODEL_ID)
        provider_ok = provider == "TypeSafe"
        mismatch = raw_response is not None and (not model_ok or not provider_ok)
        parsed, parse_ok, parse_error = None, False, None
        if isinstance(raw_response, dict) and status_code == 200 and model_ok and provider_ok:
            try:
                parsed = parsed_score(raw_response)
                parse_ok = True
            except (KeyError, TypeError, ValueError, OverflowError) as exc:
                parse_error = f"{type(exc).__name__}: {str(exc)[:400]}"
        if mismatch:
            MODEL_MISMATCH = True
            error_text = "Returned model/provider did not match pinned Jev 1.13 / TypeSafe."
        if parse_error:
            error_text = parse_error
        success = bool(status_code == 200 and model_ok and provider_ok and parse_ok and usage_valid)
        if status_code != 200 and error_text is None:
            error_text = f"Unexpected HTTP status {status_code}."
        if not usage_valid and error_text is None:
            error_text = "Response usage.input_tokens/output_tokens/cost is missing or invalid."
        record = {
            "run_id": RUN_ID, "sample_id": sample_id, "prompt_id": prompt_id,
            "prompt_version": PROMPT_VERSIONS[prompt_id], "subset": str(subset),
            "requested_model": MODEL_ID, "model": model, "provider": provider,
            "repeat_tag": str(repeat_tag), "request_hash": body_hash, "request_timestamp_utc": pd.Timestamp.now(tz="UTC").isoformat(),
            "http_status": status_code, "api_call": True, "cache_hit": False,
            "success": success, "parse_success": parse_ok, "usage_valid": usage_valid,
            "risk_score": parsed.get("risk_score") if parsed else None,
            "native_score": parsed.get("native_score") if parsed else None,
            "confidence": parsed.get("confidence") if parsed else None,
            "predicted_state": parsed.get("predicted_state") if parsed else None,
            "probabilities": parsed.get("probabilities") if parsed else None,
            "usage_input_tokens": usage_input, "usage_output_tokens": usage_output,
            "usage_cost_usd": usage_cost, "latency_seconds": latency,
            "error": error_text, "raw_response": raw_response,
        }
        append_jsonl(ATTEMPT_LEDGER_PATH, record)
        API_ATTEMPTS.append(record)
        if success:
            SUCCESSFUL_CALLS_THIS_RUN += 1
            cache_record = {**record, "repeat_tag": str(repeat_tag)}
            cache_dest = cache_path(sample_id, prompt_id, body_hash, repeat_tag)
            cache_dest.write_text(json.dumps(cache_record, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
                                  encoding="utf-8")
        else:
            API_ERRORS_THIS_RUN += 1
        if mismatch:
            print("Pinned model/provider mismatch; stopping new requests immediately.")
        return {**record, "api_call": True, "cache_hit": False, "subset": str(subset)}

    def attach_local(record, sample):
        local = {k: sample[k] for k in sample_metadata_columns}
        return {**record, **local}

    def usage_table(records):
        frame = pd.DataFrame(records)
        if frame.empty:
            return pd.DataFrame()
        rows = []
        for label, part in [("all", frame), *list(frame.groupby("prompt_id", dropna=False))]:
            row = {"prompt": label, "api_responses": int(len(part)),
                   "successful": int(part.get("success", pd.Series(dtype=bool)).fillna(False).sum()),
                   "parsed": int(part.get("parse_success", pd.Series(dtype=bool)).fillna(False).sum()),
                   "parse_rate": float(part.get("parse_success", pd.Series(dtype=bool)).fillna(False).mean())}
            for column, prefix in (("usage_input_tokens", "input_tokens"),
                                   ("usage_output_tokens", "output_tokens"),
                                   ("usage_cost_usd", "cost_usd"), ("latency_seconds", "latency_seconds")):
                values = pd.to_numeric(part.get(column, pd.Series(dtype=float)), errors="coerce").dropna().to_numpy(dtype=float)
                row[f"{prefix}_sum"] = float(np.sum(values)) if prefix != "latency_seconds" else np.nan
                row[f"{prefix}_p50"] = float(np.quantile(values, .50)) if len(values) else np.nan
                row[f"{prefix}_p95"] = float(np.quantile(values, .95, method="higher")) if len(values) else np.nan
                row[f"{prefix}_max"] = float(np.max(values)) if len(values) else np.nan
            rows.append(row)
        return pd.DataFrame(rows)

    print("Safe client ready. Cache only accepts successful, parsed, usage-complete Jev 1.13 / TypeSafe responses.")
    print("Expected API calls at most:", MAX_API_CALLS, "| expected total cost from 05a mean:", f"${estimated_cost:.4f}")

## 4. Pilot (90 requests)

The pilot takes five samples from every target-region cell in the development set. Requests are paired by sample. A model/provider mismatch stops requests immediately. Any failed parsing or usage check blocks the full run.

In [ ]:
if ARCHIVED_API_EXECUTION:
    # Pilot: 45 development samples × v4/v5.
    development_samples = selected_samples.loc[selected_samples.subset == "development"].copy()
    holdout_samples = selected_samples.loc[selected_samples.subset == "holdout"].copy()
    pilot_parts = []
    for target in targets:
        for region in REGIONS:
            cell = development_samples.loc[(development_samples.target == target) &
                                           (development_samples.region == region)].sort_values("sample_id")
            if len(cell) < 5:
                raise ValueError(f"Pilot needs 5 development rows for {target}/{region}.")
            pilot_parts.append(cell.sample(n=5, random_state=SEED))
    pilot_samples = pd.concat(pilot_parts, ignore_index=True).sort_values(
        ["target", "region", "sample_id"], kind="stable").reset_index(drop=True)
    if len(pilot_samples) != 45 or pilot_samples.sample_id.nunique() != 45:
        raise AssertionError("Pilot sample plan is not 45 distinct target-sample rows.")

    pilot_rows = []
    for _, sample in pilot_samples.iterrows():
        for prompt_id in PROMPT_IDS:
            record = invoke_jev(sample, prompt_id, "development")
            pilot_rows.append(attach_local(record, sample))
            if MODEL_MISMATCH or BUDGET_STOP:
                break
        if MODEL_MISMATCH or BUDGET_STOP:
            break
    pilot_results = pd.DataFrame(pilot_rows)
    pilot_expected = 90
    pilot_complete = len(pilot_results) == pilot_expected
    pilot_all_success = pilot_complete and bool(pilot_results.success.fillna(False).all())
    pilot_models = pilot_results.model.value_counts(dropna=False).to_dict() if len(pilot_results) else {}
    pilot_providers = pilot_results.provider.value_counts(dropna=False).to_dict() if len(pilot_results) else {}
    pilot_usage_valid = pilot_complete and bool(pilot_results.usage_valid.fillna(False).all())
    pilot_parse_rate = float(pilot_results.parse_success.fillna(False).mean()) if len(pilot_results) else 0.0
    pilot_cost_total = ledger_cost()
    pilot_projected_cost = pilot_cost_total + HISTORICAL_COST_RESERVE * max(MAX_API_CALLS - len(API_ATTEMPTS), 0)
    pilot_pass = bool(pilot_complete and pilot_all_success and pilot_usage_valid and pilot_parse_rate == 1.0 and
                      not MODEL_MISMATCH and not BUDGET_STOP and pilot_projected_cost <= HARD_COST_CAP_USD)
    pilot_results.to_csv(OUT / "pilot_results.csv", index=False)
    pilot_usage = usage_table(pilot_results.to_dict("records"))
    display(pilot_usage.round(6))
    print("Pilot expected/received:", pilot_expected, "/", len(pilot_results),
          "| parsed:", int(pilot_results.parse_success.fillna(False).sum()) if len(pilot_results) else 0,
          "| exact model distribution:", pilot_models, "| provider distribution:", pilot_providers)
    print("Pilot actual reported cost USD:", f"{pilot_cost_total:.9f}",
          "| projected cap check USD:", f"{pilot_projected_cost:.6f}",
          "| pass:", pilot_pass)
    pilot_summary = {
        "expected_requests": pilot_expected, "received_rows": int(len(pilot_results)),
        "all_requests_successful": bool(pilot_all_success), "parse_rate": pilot_parse_rate,
        "usage_fields_valid": bool(pilot_usage_valid), "returned_models": pilot_models,
        "providers": pilot_providers, "reported_cost_usd_so_far": pilot_cost_total,
        "projected_cost_usd": pilot_projected_cost, "pass": pilot_pass,
    }
    (OUT / "pilot_summary.json").write_text(json.dumps(pilot_summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    if not pilot_pass:
        print("Pilot gate failed. The full development and holdout runs will be skipped; review pilot_results.csv and api_attempts.jsonl.")

## 5. Development selection, then holdout confirmation

All 180 development samples are scored under both prompts. The candidate decision is recorded before the holdout requests. v5 passes development only when macro near_failure-minus-normal separation is higher, macro ROC-AUC and PR-AUC are each no more than 0.02 below v4, and ordered targets do not decrease.

After locking that decision, the 90 holdout samples are scored under both prompts. Final v5 retention also requires the holdout separation direction to be positive. Six development samples from the first target (two per region) are repeated once under each prompt with a distinct repeat tag.

In [ ]:
if ARCHIVED_API_EXECUTION:
    # Metrics and group-paired bootstrap used to make the development choice.
    def evaluate_primary(frame, subset_name):
        part = frame.loc[(frame.subset == subset_name) & frame.prompt_id.isin(PROMPT_IDS)].copy()
        good = part.loc[part.success.fillna(False) & part.parse_success.fillna(False)].copy()
        target_rows, region_rows = [], []
        for target in targets:
            for prompt_id in PROMPT_IDS:
                all_d = part.loc[(part.target == target) & (part.prompt_id == prompt_id)]
                d = good.loc[(good.target == target) & (good.prompt_id == prompt_id)]
                means = d.groupby("region").risk_score.mean()
                valid_regions = all(r in means.index for r in REGIONS)
                ordered = bool(means["normal"] < means["boundary"] < means["near_failure"]) if valid_regions else False
                separation = float(means["near_failure"] - means["normal"]) if valid_regions else np.nan
                binary = d.loc[d.region.isin(["normal", "near_failure"])].copy()
                labels = (binary.region == "near_failure").astype(int)
                auc = float(roc_auc_score(labels, binary.risk_score)) if labels.nunique() == 2 else np.nan
                ap = float(average_precision_score(labels, binary.risk_score)) if labels.nunique() == 2 else np.nan
                target_rows.append({
                    "subset": subset_name, "target": target, "target_label": target_labels[target],
                    "prompt_id": prompt_id, "prompt_version": PROMPT_VERSIONS[prompt_id],
                    "normal_mean": float(means.get("normal", np.nan)),
                    "boundary_mean": float(means.get("boundary", np.nan)),
                    "near_failure_mean": float(means.get("near_failure", np.nan)),
                    "separation": separation, "ordered": ordered,
                    "ROC_AUC_near_vs_normal": auc, "PR_AUC_near_vs_normal": ap,
                    "parse_rate": float(all_d.parse_success.fillna(False).mean()) if len(all_d) else np.nan,
                    "request_success_rate": float(all_d.success.fillna(False).mean()) if len(all_d) else np.nan,
                    "confidence_mean": float(d.confidence.mean()) if len(d) else np.nan,
                    "n_expected": int(len(all_d)), "n_success": int(len(d)),
                    "latency_median_seconds": float(pd.to_numeric(d.latency_seconds, errors="coerce").median()) if len(d) else np.nan,
                })
                for region in REGIONS:
                    rd = d.loc[d.region == region]
                    region_rows.append({
                        "subset": subset_name, "target": target, "target_label": target_labels[target],
                        "prompt_id": prompt_id, "region": region, "risk_mean": float(rd.risk_score.mean()) if len(rd) else np.nan,
                        "risk_std": float(rd.risk_score.std(ddof=1)) if len(rd) > 1 else np.nan,
                        "confidence_mean": float(rd.confidence.mean()) if len(rd) else np.nan,
                        "n_success": int(len(rd)),
                    })
        target_summary = pd.DataFrame(target_rows)
        region_summary = pd.DataFrame(region_rows)
        macro = target_summary.groupby(["subset", "prompt_id", "prompt_version"], as_index=False).agg(
            normal_mean=("normal_mean", "mean"), boundary_mean=("boundary_mean", "mean"),
            near_failure_mean=("near_failure_mean", "mean"), separation=("separation", "mean"),
            ordered_targets=("ordered", "sum"), ROC_AUC=("ROC_AUC_near_vs_normal", "mean"),
            PR_AUC=("PR_AUC_near_vs_normal", "mean"), parse_rate=("parse_rate", "mean"),
            request_success_rate=("request_success_rate", "mean"), confidence_mean=("confidence_mean", "mean"),
        )
        pair = good.pivot_table(
            index=["sample_id", "target", "target_label", "subset", "region", "group_id"],
            columns="prompt_id", values="risk_score", aggfunc="first").reset_index()
        if "v4" not in pair or "v5" not in pair:
            pair = pd.DataFrame(columns=["sample_id", "target", "target_label", "subset", "region", "group_id", "v4", "v5"])
            paired_region = pd.DataFrame(columns=["subset", "target", "target_label", "region", "mean_delta", "std_delta", "n"])
        else:
            pair = pair.dropna(subset=["v4", "v5"]).copy()
            pair["delta_v5_minus_v4"] = pair["v5"] - pair["v4"]
            paired_region = (pair.groupby(["subset", "target", "target_label", "region"], as_index=False)
                             .agg(mean_delta=("delta_v5_minus_v4", "mean"),
                                  std_delta=("delta_v5_minus_v4", "std"), n=("delta_v5_minus_v4", "size")))
        return target_summary, macro, region_summary, pair, paired_region

    def paired_group_bootstrap(pair, subset_name, n_boot=1000):
        if pair.empty:
            return pd.DataFrame()
        groups = np.asarray(sorted(pair.group_id.astype(str).unique()), dtype=object)
        rng = np.random.default_rng(SEED)
        draws = {"separation_delta": [], "ROC_AUC_delta": [], "PR_AUC_delta": []}
        for _ in range(n_boot):
            sampled = rng.choice(groups, size=len(groups), replace=True)
            multiplicity = pd.Series(sampled).value_counts()
            target_sep_deltas, target_auc_deltas, target_ap_deltas = [], [], []
            valid_all = True
            for target in targets:
                d = pair.loc[pair.target == target].copy()
                w = d.group_id.astype(str).map(multiplicity).fillna(0).to_numpy(dtype=float)
                means = {}
                for prompt_id in PROMPT_IDS:
                    for region in REGIONS:
                        mask = d.region.to_numpy() == region
                        denom = float(w[mask].sum())
                        means[prompt_id, region] = (float(np.average(d.loc[mask, prompt_id], weights=w[mask]))
                                                     if denom > 0 else np.nan)
                if not all(np.isfinite(means[p, r]) for p in PROMPT_IDS for r in ("normal", "near_failure")):
                    valid_all = False
                    break
                target_sep_deltas.append(
                    (means["v5", "near_failure"] - means["v5", "normal"]) -
                    (means["v4", "near_failure"] - means["v4", "normal"]))
                binary = d.region.isin(["normal", "near_failure"]).to_numpy()
                db = d.loc[binary]
                wb = w[binary]
                labels = (db.region == "near_failure").astype(int).to_numpy()
                if not len(db) or len(np.unique(labels[wb > 0])) != 2:
                    valid_all = False
                    break
                try:
                    auc4 = roc_auc_score(labels, db["v4"], sample_weight=wb)
                    auc5 = roc_auc_score(labels, db["v5"], sample_weight=wb)
                    ap4 = average_precision_score(labels, db["v4"], sample_weight=wb)
                    ap5 = average_precision_score(labels, db["v5"], sample_weight=wb)
                except ValueError:
                    valid_all = False
                    break
                target_auc_deltas.append(float(auc5 - auc4))
                target_ap_deltas.append(float(ap5 - ap4))
            if valid_all and len(target_sep_deltas) == len(targets):
                draws["separation_delta"].append(float(np.mean(target_sep_deltas)))
                draws["ROC_AUC_delta"].append(float(np.mean(target_auc_deltas)))
                draws["PR_AUC_delta"].append(float(np.mean(target_ap_deltas)))
        rows = []
        for metric, values in draws.items():
            values = np.asarray(values, dtype=float)
            rows.append({
                "subset": subset_name, "metric": metric,
                "estimate": float(np.mean(values)) if len(values) else np.nan,
                "ci95_low": float(np.quantile(values, .025)) if len(values) else np.nan,
                "ci95_high": float(np.quantile(values, .975)) if len(values) else np.nan,
                "valid_bootstrap_replicates": int(len(values)), "requested_replicates": int(n_boot),
                "bootstrap_unit": "group_id (source_file::sequence_index)",
            })
        return pd.DataFrame(rows)

    def development_candidate_decision(macro, complete):
        if not complete:
            return {"dev_complete": False, "v5_passes_development": False,
                    "reason": "Development responses are incomplete or failed strict request validation."}
        indexed = macro.set_index("prompt_id")
        if not {"v4", "v5"}.issubset(indexed.index):
            return {"dev_complete": False, "v5_passes_development": False, "reason": "Missing prompt summary."}
        b, c = indexed.loc["v4"], indexed.loc["v5"]
        conditions = {
            "macro_separation_increased": bool(c.separation > b.separation),
            "ROC_AUC_drop_less_than_0p02": bool(c.ROC_AUC >= b.ROC_AUC - 0.02),
            "PR_AUC_drop_less_than_0p02": bool(c.PR_AUC >= b.PR_AUC - 0.02),
            "ordered_targets_not_reduced": bool(c.ordered_targets >= b.ordered_targets),
        }
        return {
            "dev_complete": True, "v5_passes_development": bool(all(conditions.values())),
            "criteria": conditions,
            "v4": {k: float(b[k]) for k in ("separation", "ROC_AUC", "PR_AUC", "ordered_targets")},
            "v5": {k: float(c[k]) for k in ("separation", "ROC_AUC", "PR_AUC", "ordered_targets")},
            "rule": "Keep v5 only if all four development conditions pass; no threshold is tuned.",
        }

    print("Paired metrics and group-bootstrap helpers are ready.")

In [ ]:
if ARCHIVED_API_EXECUTION:
    # Run development, lock selection, then evaluate holdout and repeatability.
    development_rows, holdout_rows, repeat_rows = [], [], []
    dev_selection = {"dev_complete": False, "v5_passes_development": False, "reason": "Pilot gate not passed."}
    holdout_direction_reproduced = False
    final_prompt_recommendation = "v4_baseline"

    if not pilot_pass:
        print("Full comparison skipped because the pilot gate failed.")
    else:
        expected_dev_rows = len(development_samples) * len(PROMPT_IDS)
        for _, sample in development_samples.sort_values(["target", "region", "sample_id"], kind="stable").iterrows():
            for prompt_id in PROMPT_IDS:
                rec = invoke_jev(sample, prompt_id, "development")
                development_rows.append(attach_local(rec, sample))
                if MODEL_MISMATCH or BUDGET_STOP:
                    break
            if MODEL_MISMATCH or BUDGET_STOP:
                break
        development_results = pd.DataFrame(development_rows)
        development_results.to_csv(OUT / "development_results.csv", index=False)
        dev_complete = (len(development_results) == expected_dev_rows and
                        bool(development_results.success.fillna(False).all()) and
                        bool(development_results.usage_valid.fillna(False).all()))
        if len(development_results):
            dev_target_summary, dev_macro, dev_region, dev_pair, dev_paired_delta = evaluate_primary(
                development_results, "development")
            dev_ci = paired_group_bootstrap(dev_pair, "development")
            display(dev_macro.round(4))
            dev_selection = development_candidate_decision(dev_macro, dev_complete)
        else:
            dev_target_summary = dev_macro = dev_region = dev_pair = dev_paired_delta = dev_ci = pd.DataFrame()
            dev_selection = development_candidate_decision(pd.DataFrame(), False)
        dev_selection["selection_locked_utc"] = pd.Timestamp.now(tz="UTC").isoformat()
        (OUT / "development_selection.json").write_text(
            json.dumps(dev_selection, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
        print("Development selection locked before holdout:", dev_selection)
        if dev_complete and not MODEL_MISMATCH and not BUDGET_STOP:
            expected_holdout_rows = len(holdout_samples) * len(PROMPT_IDS)
            for _, sample in holdout_samples.sort_values(["target", "region", "sample_id"], kind="stable").iterrows():
                for prompt_id in PROMPT_IDS:
                    rec = invoke_jev(sample, prompt_id, "holdout")
                    holdout_rows.append(attach_local(rec, sample))
                    if MODEL_MISMATCH or BUDGET_STOP:
                        break
                if MODEL_MISMATCH or BUDGET_STOP:
                    break
            holdout_results = pd.DataFrame(holdout_rows)
            holdout_results.to_csv(OUT / "holdout_results.csv", index=False)
            holdout_complete = (len(holdout_results) == expected_holdout_rows and
                                bool(holdout_results.success.fillna(False).all()) and
                                bool(holdout_results.usage_valid.fillna(False).all()))
            if len(holdout_results):
                hold_target_summary, hold_macro, hold_region, hold_pair, hold_paired_delta = evaluate_primary(
                    holdout_results, "holdout")
                hold_ci = paired_group_bootstrap(hold_pair, "holdout")
                display(hold_macro.round(4))
                if holdout_complete and {"v4", "v5"}.issubset(set(hold_macro.prompt_id)):
                    hm = hold_macro.set_index("prompt_id")
                    holdout_direction_reproduced = bool(hm.loc["v5", "separation"] > hm.loc["v4", "separation"])
            else:
                hold_target_summary = hold_macro = hold_region = hold_pair = hold_paired_delta = hold_ci = pd.DataFrame()
                holdout_complete = False
            if dev_selection.get("v5_passes_development") and holdout_complete and holdout_direction_reproduced:
                final_prompt_recommendation = "v5_candidate"
            else:
                final_prompt_recommendation = "v4_baseline"
            print("Holdout separation direction reproduced:", holdout_direction_reproduced,
                  "| final exploratory recommendation:", final_prompt_recommendation)

            first_target = targets[0]
            repeat_samples = (development_samples.loc[development_samples.target == first_target]
                              .sort_values(["region", "sample_id"], kind="stable")
                              .groupby("region", sort=False).head(2))
            if len(repeat_samples) == 6 and holdout_complete:
                primary_lookup = development_results.set_index(["sample_id", "prompt_id"])
                for _, sample in repeat_samples.iterrows():
                    for prompt_id in PROMPT_IDS:
                        rec = invoke_jev(sample, prompt_id, "development", repeat_tag="repeat_1")
                        repeated = attach_local(rec, sample)
                        baseline = primary_lookup.loc[(str(sample.sample_id), prompt_id)]
                        baseline_risk = float(baseline.risk_score) if pd.notna(baseline.risk_score) else np.nan
                        repeat_risk = float(rec["risk_score"]) if rec.get("risk_score") is not None else np.nan
                        pair_values = np.asarray([baseline_risk, repeat_risk], dtype=float)
                        pair_values = pair_values[np.isfinite(pair_values)]
                        repeated.update({
                            "baseline_risk": baseline_risk, "repeat_risk": repeat_risk,
                            "risk_delta_repeat_minus_primary": repeat_risk - baseline_risk
                            if np.isfinite(repeat_risk) and np.isfinite(baseline_risk) else np.nan,
                            "paired_sd": float(pair_values.std(ddof=1)) if len(pair_values) == 2 else np.nan,
                            "paired_range": float(np.ptp(pair_values)) if len(pair_values) == 2 else np.nan,
                        })
                        repeat_rows.append(repeated)
                        if MODEL_MISMATCH or BUDGET_STOP:
                            break
                    if MODEL_MISMATCH or BUDGET_STOP:
                        break
            repeat_results = pd.DataFrame(repeat_rows)
            repeat_results.to_csv(OUT / "repeatability.csv", index=False)
            print("Repeat calls:", len(repeat_results), "| max design: 12.")
        else:
            holdout_complete = False
            hold_target_summary = hold_macro = hold_region = hold_pair = hold_paired_delta = hold_ci = pd.DataFrame()
            print("Holdout calls skipped because the development set was incomplete or a hard stop was triggered.")
            final_prompt_recommendation = "v4_baseline"

## 6. Paired evaluation, usage, plots, and limits

Metrics are computed separately for development and holdout. Region risk means include all three TTF strata; ROC-AUC and PR-AUC use near_failure versus normal. Macro values average the three target-level metrics. The paired delta is v5 minus v4 for the same sample. Group bootstrap confidence intervals resample source_file::sequence_index groups.

The fresh v4 1× result is also compared by target and region with the old 05a Robust rows. Those samples are disjoint by design, so this historical comparison is an unpaired cohort-level change. Jev scores and confidence remain decision-support outputs, not measured fault truth.

In [ ]:
if ARCHIVED_API_EXECUTION:
    # Aggregate artifacts, bootstrap intervals, repeatability, plots, and run summary.
    if "holdout_results" not in globals():
        holdout_results = pd.DataFrame(holdout_rows)
    if "development_results" not in globals():
        development_results = pd.DataFrame(development_rows)
    primary_results = pd.concat([development_results, holdout_results], ignore_index=True, sort=False)
    repeat_results = pd.DataFrame(repeat_rows)

    target_parts, macro_parts, region_parts, paired_parts, ci_parts = [], [], [], [], []
    for subset_name, frame in (("development", development_results), ("holdout", holdout_results)):
        if len(frame):
            ts, mc, rs, pair, pdlt = evaluate_primary(frame, subset_name)
            target_parts.append(ts); macro_parts.append(mc); region_parts.append(rs); paired_parts.append(pdlt)
            ci_parts.append(paired_group_bootstrap(pair, subset_name))
    target_comparison = pd.concat(target_parts, ignore_index=True) if target_parts else pd.DataFrame()
    macro_comparison = pd.concat(macro_parts, ignore_index=True) if macro_parts else pd.DataFrame()
    region_risk_summary = pd.concat(region_parts, ignore_index=True) if region_parts else pd.DataFrame()
    paired_delta_by_region = pd.concat(paired_parts, ignore_index=True) if paired_parts else pd.DataFrame()
    bootstrap_ci = pd.concat(ci_parts, ignore_index=True) if ci_parts else pd.DataFrame()

    # Unpaired cohort-level comparison against the historical 05a Robust responses.
    fresh_control_vs_05a = pd.DataFrame()
    if len(primary_results):
        new_v4 = primary_results.loc[(primary_results.prompt_id == "v4") & primary_results.success.fillna(False)]
        new_means = new_v4.groupby(["target", "region"], as_index=False).agg(
            new_v4_1x_mean=("risk_score", "mean"), new_n=("risk_score", "size"))
        old_means = old_robust.groupby(["target", "region"], as_index=False).agg(
            old_05a_robust_mean=("risk_score", "mean"), old_n=("risk_score", "size"))
        fresh_control_vs_05a = new_means.merge(old_means, on=["target", "region"], how="outer")
        fresh_control_vs_05a["new_minus_old_unpaired"] = (
            fresh_control_vs_05a.new_v4_1x_mean - fresh_control_vs_05a.old_05a_robust_mean)
        fresh_control_vs_05a["comparison_note"] = "Different sample rows; unpaired cohort-level comparison."

    attempts_frame = pd.DataFrame(API_ATTEMPTS)
    if len(attempts_frame):
        usage_summary = usage_table(API_ATTEMPTS)
        model_provider_distribution = (attempts_frame.groupby(["prompt_id", "model", "provider"], dropna=False)
                                       .size().rename("api_responses").reset_index())
    else:
        usage_summary = pd.DataFrame()
        model_provider_distribution = pd.DataFrame()
    if len(primary_results):
        model_provider_distribution_cached = (
            primary_results.loc[primary_results.model.notna()].groupby(["prompt_id", "model", "provider"], dropna=False)
            .size().rename("primary_response_rows").reset_index())
    else:
        model_provider_distribution_cached = pd.DataFrame()

    target_comparison.to_csv(OUT / "target_comparison.csv", index=False)
    macro_comparison.to_csv(OUT / "macro_comparison.csv", index=False)
    region_risk_summary.to_csv(OUT / "region_risk_summary.csv", index=False)
    paired_delta_by_region.to_csv(OUT / "paired_delta_by_region.csv", index=False)
    bootstrap_ci.to_csv(OUT / "group_paired_bootstrap_ci.csv", index=False)
    fresh_control_vs_05a.to_csv(OUT / "fresh_v4_1x_vs_05a_robust_unpaired.csv", index=False)
    usage_summary.to_csv(OUT / "usage_summary.csv", index=False)
    model_provider_distribution.to_csv(OUT / "model_provider_distribution.csv", index=False)
    model_provider_distribution_cached.to_csv(OUT / "primary_response_model_provider_distribution.csv", index=False)
    if len(primary_results):
        primary_results.to_csv(OUT / "all_results.csv", index=False)
    if len(repeat_results):
        repeat_results.to_csv(OUT / "repeatability.csv", index=False)

    # Visualize response scores without exposing numeric input features.
    if len(primary_results) and primary_results.risk_score.notna().any():
        fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
        for row_i, subset_name in enumerate(("development", "holdout")):
            for col_i, target in enumerate(targets):
                ax = axes[row_i, col_i]
                for prompt_id in PROMPT_IDS:
                    d = primary_results.loc[(primary_results.subset == subset_name) &
                                            (primary_results.target == target) &
                                            (primary_results.prompt_id == prompt_id) &
                                            primary_results.risk_score.notna()]
                    means = d.groupby("region").risk_score.mean().reindex(REGIONS)
                    ax.plot(range(3), means.to_numpy(), marker="o", label=prompt_id)
                ax.set_xticks(range(3), ["Normal", "Boundary", "Near failure"])
                ax.set_ylim(-0.03, 1.03)
                ax.set_title(f"{subset_name}: {target_labels[target]}")
                ax.grid(axis="y", alpha=.25)
        axes[0, 0].set_ylabel("Jev 1.13 risk score")
        axes[1, 0].set_ylabel("Jev 1.13 risk score")
        axes[0, 2].legend(frameon=False)
        fig.tight_layout()
        fig.savefig(OUT / "risk_by_region_development_holdout.png", dpi=160, bbox_inches="tight")
        display(fig)
        plt.close(fig)

    if len(repeat_results):
        repeat_summary = (repeat_results.groupby("prompt_id", as_index=False).agg(
            repeats=("sample_id", "size"),
            mean_abs_risk_delta=("risk_delta_repeat_minus_primary", lambda s: float(np.nanmean(np.abs(s)))),
            mean_paired_sd=("paired_sd", "mean"), mean_paired_range=("paired_range", "mean"),
            parse_rate=("parse_success", "mean")))
    else:
        repeat_summary = pd.DataFrame()
    repeat_summary.to_csv(OUT / "repeatability_summary.csv", index=False)

    run_attempts = API_ATTEMPTS[RUN_ATTEMPT_START:]
    run_attempt_frame = pd.DataFrame(run_attempts)
    run_successful = int(run_attempt_frame.success.fillna(False).sum()) if len(run_attempt_frame) else 0
    run_parse_errors = int((~run_attempt_frame.parse_success.fillna(False)).sum()) if len(run_attempt_frame) else 0
    run_models = run_attempt_frame.model.value_counts(dropna=False).to_dict() if len(run_attempt_frame) else {}
    run_providers = run_attempt_frame.provider.value_counts(dropna=False).to_dict() if len(run_attempt_frame) else {}
    run_input_tokens = int(pd.to_numeric(run_attempt_frame.usage_input_tokens, errors="coerce").fillna(0).sum()) if len(run_attempt_frame) else 0
    run_output_tokens = int(pd.to_numeric(run_attempt_frame.usage_output_tokens, errors="coerce").fillna(0).sum()) if len(run_attempt_frame) else 0
    run_reported_cost = float(pd.to_numeric(run_attempt_frame.usage_cost_usd, errors="coerce").fillna(0).sum()) if len(run_attempt_frame) else 0.0
    run_cached_primary_rows = int(primary_results.cache_hit.fillna(False).sum()) if len(primary_results) else 0
    full_primary_grid_complete = (len(primary_results) == len(selected_samples) * 2 and
                                  bool(primary_results.success.fillna(False).all()))
    repeat_summary.to_csv(OUT / "repeatability_summary.csv", index=False)
    display(target_comparison.round(4))
    display(macro_comparison.round(4))
    display(bootstrap_ci.round(4))
    display(usage_summary.round(6))
    display(repeat_summary.round(5))
    print("Actual API calls this execution:", API_CALLS_THIS_RUN,
          "| successful:", run_successful, "| current-run cache hits:", CACHE_HITS_THIS_RUN,
          "| primary result cache hits:", run_cached_primary_rows)
    print("Returned model/provider distributions:", run_models, run_providers,
          "| parsing errors:", run_parse_errors)
    print("Reported usage this execution: input", run_input_tokens, "| output", run_output_tokens,
          "| cost USD", f"{run_reported_cost:.9f}", "| hard cap USD", f"{HARD_COST_CAP_USD:.2f}")
    print("Development/holdout group overlap:", len(dev_groups & hold_groups),
          "| full primary grid complete:", full_primary_grid_complete,
          "| development selected v5:", bool(dev_selection.get("v5_passes_development")),
          "| holdout direction reproduced:", holdout_direction_reproduced,
          "| exploratory recommendation:", final_prompt_recommendation)

    config = {
        "experiment": "05b Jev 1.13 paired prompt optimization",
        "run_id": RUN_ID, "seed": SEED,
        "requested_model": MODEL_ID, "endpoint": DECISIONS_URL, "api_docs": DOCS_URL,
        "prompt_versions": PROMPT_VERSIONS, "scale": "05a train-only Robust, 1x fixed",
        "feature_count": len(features), "analysis_group": split_meta["group_definition"],
        "fixed_validation_groups": len(validation_group_ids),
        "prompt_development_groups": len(DEVELOPMENT_GROUPS),
        "prompt_holdout_groups": len(HOLDOUT_GROUPS),
        "sample_count": int(len(selected_samples)),
        "sample_counts_by_subset": {str(k): int(v) for k, v in selected_samples.groupby("subset").size().items()},
        "sample_counts_by_target_region": selected_samples.groupby(["subset", "target", "region"]).size().rename("n").reset_index().to_dict(orient="records"),
        "prior_05a_sample_rows_excluded": len(old_sample_keys),
        "group_overlap_development_holdout": len(dev_groups & hold_groups),
        "pilot": pilot_summary, "development_selection": dev_selection,
        "holdout_direction_reproduced": bool(holdout_direction_reproduced),
        "final_exploratory_recommendation": final_prompt_recommendation,
        "max_api_calls": MAX_API_CALLS, "api_calls_this_execution": API_CALLS_THIS_RUN,
        "successful_api_responses_this_execution": run_successful,
        "cache_hits_this_execution": CACHE_HITS_THIS_RUN, "parse_errors_this_execution": run_parse_errors,
        "returned_model_counts_this_execution": run_models, "provider_counts_this_execution": run_providers,
        "input_tokens_reported_this_execution": run_input_tokens,
        "output_tokens_reported_this_execution": run_output_tokens,
        "reported_cost_usd_this_execution": run_reported_cost,
        "hard_cost_cap_usd": HARD_COST_CAP_USD, "budget_stop_triggered": bool(BUDGET_STOP),
        "credential_environment_names": list(EXPECTED_API_KEY_NAMES), "credential_saved": False,
        "raw_phm_data_saved": False, "request_feature_values_saved": False,
        "candidate_scope": "Small paired prompt comparison; Jev outputs are not ground truth.",
    }
    (OUT / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
    summary_lines = [
        "# 05b Jev prompt optimization run summary", "",
        f"- Sample rows: {len(selected_samples)} ({len(development_samples)} development, {len(holdout_samples)} holdout); prior 05a rows excluded: {len(old_sample_keys)}.",
        f"- Group overlap between development and holdout: {len(dev_groups & hold_groups)}.",
        f"- Pilot passed: {pilot_pass}; development complete: {dev_selection.get('dev_complete')}; v5 passed development: {dev_selection.get('v5_passes_development')}.",
        f"- Holdout separation direction reproduced: {holdout_direction_reproduced}; exploratory recommendation: {final_prompt_recommendation}.",
        f"- API attempts this execution: {API_CALLS_THIS_RUN}; successful responses: {run_successful}; current-run cache hits: {CACHE_HITS_THIS_RUN}; parse errors: {run_parse_errors}.",
        f"- Returned models: {run_models}; providers: {run_providers}.",
        f"- OpenRouter reported tokens: input {run_input_tokens}, output {run_output_tokens}; reported cost USD {run_reported_cost:.9f} (cap {HARD_COST_CAP_USD:.2f}).",
        "- Both prompts used the same 20 train-only Robust-transformed numeric features at 1x. No model was trained or substituted.",
        "- Evaluation is a small grouped prompt experiment; TTF strata are balanced by design, Jev outputs are decision support, and the old 05a comparison is unpaired.",
    ]
    (OUT / "run_summary.md").write_text("\n".join(summary_lines) + "\n", encoding="utf-8")
    print("Saved experiment artifacts under:", OUT)
    print("Summary:", OUT / "run_summary.md")

## Interpretation limits

This is a small, balanced prompt comparison on a group-split portion of the existing validation set. The balanced strata do not reflect operational fault prevalence. Holdout confirms prompt direction only after development selection; it is not an untouched external test set. ROC-AUC and PR-AUC are descriptive, confidence is not calibration, and Jev scores are not labels or ground truth.

No threshold was adjusted. v5 is only an exploratory candidate if it passes the stated development rule and the holdout separation direction repeats; otherwise keep v4 and the fixed 1× Robust input.

## 7. Offline diagnostic audit and exploratory three-band thresholds

This section reuses the saved 05b responses and the processed 02 window features. It reconstructs all 540 primary request hashes, audits physical-window and group membership, compares the disjoint 05a/05b cohorts, and selects two shared v4 thresholds on development groups only. Its TTF bands are proxy labels; the earlier holdout has already been viewed for prompt selection. No API credential or network access is used.


In [ ]:
from pathlib import Path
import sys
ROOT_OFFLINE = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p / "artifacts/jev_prompt_optimization/config.json").is_file()), None)
if ROOT_OFFLINE is None:
    raise FileNotFoundError("05b saved artifacts and project root are required for offline diagnostics.")
site = ROOT_OFFLINE / ".venv" / "Lib" / "site-packages"
if site.is_dir():
    sys.path.insert(0, str(site))
sys.path.insert(0, str(ROOT_OFFLINE / "src"))
from IPython.display import display, Image
from fabjudge.jev_prompt_diagnostics import run_diagnostics
offline_summary = run_diagnostics(ROOT_OFFLINE, bootstrap_replicates=1000)
diagnostics_dir = ROOT_OFFLINE / "artifacts" / "jev_prompt_optimization" / "diagnostics"
print((diagnostics_dir / "summary.md").read_text(encoding="utf-8"))
import pandas as pd
display(pd.read_csv(diagnostics_dir / "sample_group_audit.csv"))
display(pd.read_csv(diagnostics_dir / "threshold_metrics.csv"))
display(pd.read_csv(diagnostics_dir / "threshold_class_metrics.csv"))
display(pd.read_csv(diagnostics_dir / "threshold_cv_folds.csv"))
display(pd.read_csv(diagnostics_dir / "threshold_group_bootstrap_ci.csv"))
display(pd.read_csv(diagnostics_dir / "paired_group_bootstrap_observed_and_ci.csv"))
display(Image(filename=str(diagnostics_dir / "risk_score_distributions.png")))
display(Image(filename=str(diagnostics_dir / "paired_prompt_deltas.png")))
display(Image(filename=str(diagnostics_dir / "feature_magnitude_cohorts.png")))


## Offline interpretation limits

The two fitted cutoffs describe score bands against balanced TTF proxy labels. They do not calibrate current equipment state, failure probability, or JEV confidence. Group folds and bootstrap samples measure instability within the saved development groups. The reused holdout is secondary evidence; operational thresholds require new, unused groups and measured present-state labels. The v4 prompt remains the saved 05b choice, without a claim of operational gate performance.


## Noul follow-up preflight

This section audits the saved 05a, scale-sensitivity, 05b, and diagnostics artifacts before any new API request. Historical results are summarized separately and remain read-only. This follow-up has a distinct output folder, model/provider checks, and a new budget of 300 requests and $0.05 reported cost.

The planned request pins `typesafe/jev-1.13` at the OpenRouter Decisions API and places the exact saved v4 Score question and a Noul yes/no evidence question in one request over the same 20 numeric features. The candidate rule and threshold-selection procedure are fixed before development responses. TTF bands are evaluation surrogates, not current equipment-state truth. The code below sends no request unless a new balanced nine-cell development/evaluation design can be formed from unused groups.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, str(Path.cwd() / '.venv' / 'Lib' / 'site-packages'))
import pandas as pd
ROOT = Path.cwd().resolve()
ART = ROOT / 'artifacts'
OUT_DIR = ART / 'jev_prompt_optimization' / 'noul_followup'
OUT_DIR.mkdir(parents=True, exist_ok=True)
base = json.loads((ART / 'jev_tut' / 'config.json').read_text(encoding='utf-8'))
old5b = json.loads((ART / 'jev_prompt_optimization' / 'config.json').read_text(encoding='utf-8'))
old_questions = json.loads((ART / 'jev_prompt_optimization' / 'decision_questions.json').read_text(encoding='utf-8'))
v4_question = json.loads((ART / 'jev_tut' / 'prompt_v4.txt').read_text(encoding='utf-8').split('question=\n', 1)[1])
assert old_questions['v4'] == v4_question
assert base['model_id'] == old5b['requested_model'] == 'typesafe/jev-1.13'
assert len(base['feature_columns']) == 20
assert old5b['api_calls_this_execution'] == old5b['max_api_calls'] == 552
assert old5b['development_selection']['v5_passes_development'] is False
assert old5b['final_exploratory_recommendation'] == 'v4_baseline'

# Historical comparison is read-only and reported separately from the new run.
scale_old = pd.read_csv(ART / 'jev_scale_sensitivity' / 'macro_comparison.csv').set_index('representation')
scale_ext = pd.read_csv(ART / 'jev_scale_sensitivity' / 'extreme_0p1_10x' / 'macro_comparison.csv').set_index('arm')
scale_run = json.loads((ART / 'jev_scale_sensitivity' / 'extreme_0p1_10x' / 'summary.json').read_text(encoding='utf-8'))
assert scale_run['new_request_count'] == 135
for metric in ['separation', 'ROC_AUC', 'PR_AUC']:
    assert float(scale_old.loc['down_0p5', metric]) < float(scale_old.loc['control_1p0', metric])
    assert float(scale_old.loc['up_2p0', metric]) < float(scale_old.loc['control_1p0', metric])
    assert float(scale_ext.loc['0.1x new', metric]) < float(scale_ext.loc['1x fresh', metric])
    assert float(scale_ext.loc['10x new', metric]) < float(scale_ext.loc['1x fresh', metric])
old05a = pd.read_csv(ART / 'jev_tut' / 'selected_samples.csv', usecols=['source_file', 'sequence_index'])
new05b = pd.read_csv(ART / 'jev_prompt_optimization' / 'sample_audit.csv', usecols=['group_id', 'subset'])
old05a['group_id'] = old05a.source_file.astype(str) + '::' + old05a.sequence_index.astype(str)
g05a, g05b = set(old05a.group_id), set(new05b.group_id)
history = {
    'analysis_unit':'100 observations/window; stride 50; adjacent windows overlap',
    'group_unit':'source_file::sequence_index', 'features':20,
    'preprocessing':'train-only Robust, fixed 1x', '05a_rows':45, '05a_unique_groups':len(g05a),
    '05b_rows':old5b['sample_count'], '05b_development_rows':old5b['sample_counts_by_subset']['development'],
    '05b_holdout_rows_already_observed':old5b['sample_counts_by_subset']['holdout'],
    '05b_groups':len(g05b), '05a_05b_group_intersection':len(g05a & g05b), '05a_05b_group_union':len(g05a | g05b),
    '05b_requests_and_cap_exhausted':old5b['api_calls_this_execution'],
    '05b_cache_hits':old5b['cache_hits_this_execution'], '05b_input_tokens':old5b['input_tokens_reported_this_execution'],
    '05b_output_tokens':old5b['output_tokens_reported_this_execution'], '05b_cost_usd':old5b['reported_cost_usd_this_execution'],
    '05b_v5_passed_development':old5b['development_selection']['v5_passes_development'],
    '05b_retained_prompt':old5b['final_exploratory_recommendation'],
    'prior_thresholds_operationally_adopted':False,
    'scale_0p5_old_separation':float(scale_old.loc['down_0p5','separation']),
    'scale_1x_old_separation':float(scale_old.loc['control_1p0','separation']),
    'scale_2x_old_separation':float(scale_old.loc['up_2p0','separation']),
    'scale_0p1_new_separation':float(scale_ext.loc['0.1x new','separation']),
    'scale_1x_fresh_separation':float(scale_ext.loc['1x fresh','separation']),
    'scale_10x_new_separation':float(scale_ext.loc['10x new','separation']),
    'extreme_factors_compared_to_same_batch_fresh_1x':True,
    'old_vs_fresh_1x_mean_abs_risk_change_not_scale_effect':float(scale_run['fresh_1_vs_historical_1_mean_abs_risk_change']),
}
(OUT_DIR / 'historical_verification.json').write_text(json.dumps(history, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
historical_rows = [
    {'source':'05a + scale 0.5x/1x/2x','finding':'0.5x and 2x were below the same-batch 1x comparison on separation, ROC-AUC, and PR-AUC.'},
    {'source':'fresh 0.1x/1x/10x batch','finding':'0.1x and 10x were below that batch fresh 1x on separation, ROC-AUC, and PR-AUC. Historical 1x vs fresh 1x is not a scale-effect estimate.'},
    {'source':'05b','finding':'v5 failed the locked development rule; v4 remained exploratory baseline. Prior three-band thresholds remain candidates, not an operating decision.'},
]
print('HISTORICAL RESULTS - read-only, separate from this follow-up')
print(pd.DataFrame(historical_rows).to_string(index=False))
print('Historical facts:', json.dumps(history, ensure_ascii=False, sort_keys=True))
print('Saved 05b development/holdout prompt metrics:')
print(pd.read_csv(ART / 'jev_prompt_optimization' / 'macro_comparison.csv').to_string(index=False))

# Audit candidate groups and 05b group use before constructing any new sample.
avail = pd.read_csv(ART / 'jev_prompt_optimization' / 'sample_availability_audit.csv')
used = pd.read_csv(ART / 'jev_prompt_optimization' / 'diagnostics' / 'sample_group_audit.csv')
assert len(avail) == 9 and len(used) == 18
used_groups = used.pivot(index=['target','region'], columns='subset', values='unique_groups').reset_index()
used_samples = used.pivot(index=['target','region'], columns='subset', values='samples').reset_index()
used_groups = used_groups.rename(columns={'development':'dev_groups_used_by_05b','holdout':'eval_groups_used_by_05b'})
used_samples = used_samples.rename(columns={'development':'dev_samples_used_by_05b','holdout':'eval_samples_used_by_05b'})
audit = avail.merge(used_groups, on=['target','region'], validate='one_to_one').merge(used_samples, on=['target','region'], validate='one_to_one')
audit['dev_groups_unseen_by_05b_upper_bound'] = audit.development_groups_available - audit.dev_groups_used_by_05b
audit['eval_groups_unseen_by_05b_upper_bound'] = audit.holdout_groups_available - audit.eval_groups_used_by_05b
if (audit[['dev_groups_unseen_by_05b_upper_bound','eval_groups_unseen_by_05b_upper_bound']] < 0).any().any():
    raise ValueError('A sampled 05b group was absent from its audited candidate group pool.')
# The remaining counts are upper bounds after removing 05b groups; excluding any additional 05a groups can only lower them.
rare = audit.loc[(audit.target == 'ttf_flowcool_leak_seconds') & (audit.region == 'near_failure')].iloc[0]
assert int(rare.development_groups_available) == int(rare.dev_groups_used_by_05b) == 6
assert int(rare.holdout_groups_available) == int(rare.eval_groups_used_by_05b) == 2
assert int(rare.dev_groups_unseen_by_05b_upper_bound) == int(rare.eval_groups_unseen_by_05b_upper_bound) == 0
dev_cell_cap = int(audit.dev_groups_unseen_by_05b_upper_bound.min())
eval_cell_cap = int(audit.eval_groups_unseen_by_05b_upper_bound.min())
assert min(dev_cell_cap, eval_cell_cap) == 0
audit.to_csv(OUT_DIR / 'feasibility_audit.csv', index=False)
status = {
    'sample_audit_source':'saved validation availability after excluding 05a physical sample rows',
    'group_definition':'source_file::sequence_index',
    'sample_unit':'100 observations/window; stride 50; neighboring windows overlap',
    'prior_groups':{'05a':len(g05a),'05b':len(g05b),'shared':len(g05a & g05b),'union':len(g05a | g05b)},
    'leak_near_failure':{'dev_candidate_groups':int(rare.development_groups_available),'dev_groups_used_by_05b':int(rare.dev_groups_used_by_05b),'eval_candidate_groups':int(rare.holdout_groups_available),'eval_groups_used_by_05b':int(rare.eval_groups_used_by_05b),'unused_groups':0},
    'new_balanced_nine_cell_design':{'dev_min_unused_group_upper_bound':dev_cell_cap,'eval_min_unused_group_upper_bound':eval_cell_cap,'samples_per_cell':0,'feasible':False},
    'new_group_cv_feasible':False,
    'reason':'The leak/near_failure cell has no unused groups in the existing validation pool. The 6 development-candidate and 2 evaluation-candidate groups were all sampled by 05b; excluding more 05a groups cannot increase availability.'
}
(OUT_DIR / 'feasibility_status.json').write_text(json.dumps(status, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('\nNEW NOUL FOLLOW-UP - pre-API availability audit')
cols = ['target','region','eligible_after_05a_exclusion','development_groups_available','dev_groups_used_by_05b','dev_groups_unseen_by_05b_upper_bound','holdout_groups_available','eval_groups_used_by_05b','eval_groups_unseen_by_05b_upper_bound']
print(audit[cols].to_string(index=False))
print('Leak/near_failure: 6/6 development groups and 2/2 evaluation-candidate groups were used by 05b; unused groups = 0.')
print('Equal nine-cell design capacity upper bounds: development',dev_cell_cap,'groups/cell; evaluation',eval_cell_cap,'groups/cell.')

# Exact questions and rules are saved even though no development response exists.
question_noul = {
    'type':'noul',
    'instructions':'Using only the numeric measurements in state.features, do the measurements as a whole provide sufficient evidence of a current abnormal equipment condition?',
    'criteria':{
        'true':'The combined measurements support a present abnormal condition.',
        'false':'The combined measurements do not provide sufficient evidence of a present abnormal condition.'
    }
}
features = base['feature_columns']
request_shape = {'model':'typesafe/jev-1.13','state':{'features':{f:0.0 for f in features}},'questions':{'score':v4_question,'noul':question_noul}}
assert set(request_shape) == {'model','state','questions'} and set(request_shape['state']) == {'features'}
assert len(request_shape['state']['features']) == 20 and set(request_shape['questions']) == {'score','noul'}
request_audit = {
    'model':'typesafe/jev-1.13','endpoint':'https://openrouter.ai/api/alpha/decisions',
    'top_level_fields':['model','state','questions'],'state_fields':['features'],'feature_count':20,
    'questions':{'score':'exact question object from artifacts/jev_tut/prompt_v4.txt','noul':question_noul},
    'schema_only_check':'passed in memory with numeric placeholders; placeholders were not sent or saved',
    'sample_payloads_built':0,'requests_sent':0,
    'local_sample_ids_targets_truth_ttf_regions_filenames_group_ids_in_body':False,
    'credentials_read_or_saved':False,'raw_feature_values_saved':False,
    'cache_key_fields':['local_sample_id','model','question_version','canonical_request_sha256'],
    'failed_response_reused_as_success_cache':False,
    'response_validation':{'model_prefix':'typesafe/jev-1.13','provider':'TypeSafe','usage_fields':['input_tokens','output_tokens','cost']}
}
rule_design = {
    'version':'noul-current-condition-v1 + exact-v4-score','model':'typesafe/jev-1.13',
    'score_question':v4_question,'noul_question':question_noul,
    'interpretation':'Noul and Score are decision-support outputs, not fault probabilities or ground truth.',
    'noul_0p5':'A split yes/no evidence judgement, not severity, the TTF boundary, a middle Score, or Jev Uncertain.',
    'noul_binary_audit':'Use 0.5 for the yes/no split in normal-vs-near_failure surrogate discrimination; exclude TTF boundary from that two-class contrast and report it separately.',
    'candidate_rule':{'noul_missing':'review','noul_below_0p4':'auto normal/low gate','noul_0p4_to_0p6_inclusive':'review required; not a medium-risk class','noul_above_0p6':'apply same-request v4 Score thresholds','score_missing':'review for both methods','score_ties':'score <= t1 => normal/low; t1 < score <= t2 => boundary/middle; score > t2 => near_failure/high'},
    'thresholds':{'numeric_t1_t2':'selected only on development groups; none selected because no valid development sample','search':'ordered t1<t2 from 0/1 endpoints and midpoints between unique training scores','objective':'maximize mean per-target 3-class balanced accuracy; review counts as incorrect','ties':'within 1e-12 prefer higher macro-F1, then lexicographically smaller (t1,t2)','cv':'up to 3 GroupKFold folds only when all nine cells appear in every split; choose thresholds in fold-training groups only','evaluation':'lock after development and apply once to untouched groups'},
    'candidate_metrics':'per-target/macro balanced accuracy, macro-F1, confusion matrices, class precision/recall, normal-high rate, near_failure-low rate, missed near_failure at Noul gate, review fraction, automatic coverage and automatic-sample error rate',
    'bootstrap':'paired by group; resample whole groups with all rows and both method predictions; point estimates from original sample',
    'limits':'TTF bands are surrogate labels; balanced precision and PR-AUC do not estimate operating prevalence.',
    'planned_design':{'max_samples_per_target_region_per_subset':15,'cells_per_subset':9,'planned_requests':270,'pilot':'up to 5 development samples per cell, included in 270','request_cap':300,'reported_cost_cap_usd':0.05}
}
(OUT_DIR / 'request_body_audit.json').write_text(json.dumps(request_audit, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(OUT_DIR / 'question_and_rule_design.json').write_text(json.dumps(rule_design, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
run_status = {
    'status':'preflight_blocked_no_new_api_requests','reason':status['reason'],
    'requested_model':'typesafe/jev-1.13','endpoint':'https://openrouter.ai/api/alpha/decisions',
    'new_budget':{'max_requests':300,'hard_reported_cost_usd':0.05},
    'new_run_usage':{'requests':0,'successful_responses':0,'cache_hits':0,'parse_errors':0,'input_tokens':0,'output_tokens':0,'reported_cost_usd':0.0,'returned_models':{},'providers':{},'usage_fields_received':False},
    'pilot':'not run: no balanced nine-cell new-group sample','development':'not run','group_cv':'not feasible for new sample','evaluation':'not run','plots':'not generated; no responses',
    'noul_missed_near_failure_count':None,'review_fraction':None,'automatic_coverage':None,
    'conclusion':'No comparative estimate; exploratory candidate only; operational adoption deferred.',
    'prior_05b_usage_separate':{'requests':old5b['api_calls_this_execution'],'cache_hits':old5b['cache_hits_this_execution'],'input_tokens':old5b['input_tokens_reported_this_execution'],'output_tokens':old5b['output_tokens_reported_this_execution'],'reported_cost_usd':old5b['reported_cost_usd_this_execution']}
}
(OUT_DIR / 'pilot_status.json').write_text(json.dumps({'status':'not_run','reason':run_status['pilot']},ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
(OUT_DIR / 'run_status.json').write_text(json.dumps(run_status,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
summary_lines = [
    '# Noul follow-up preflight summary','',
    '## Historical results (read-only; separate from this follow-up)','',
    '- Analysis unit: length 100, stride 50, overlapping windows; group: source_file::sequence_index. The same 20 train-only Robust features at 1x were used.',
    '- Scale comparisons: 0.5x/2x were below their old 1x batch; 0.1x/10x were below the fresh 1x in their own batch. Old 1x vs fresh 1x is not interpreted as a scale effect.',
    '- 05b v5 failed development selection; v4 was retained. Three-band thresholds remain exploratory, not adopted for operation.',
    '- The earlier 05b run exhausted 552 requests. Existing responses, cache, settings, selection and diagnostics were not modified.','',
    '## New Noul follow-up','',
    '- The validation pool has no unused leak/near_failure groups after 05b: all 6 development candidates and 2 evaluation candidates were sampled. The 05a group exclusion can only reduce availability.',
    '- A balanced nine-cell development/evaluation sample has capacity 0 per cell. New group-preserving CV and untouched evaluation are infeasible from this pool.',
    '- No API request was sent. Separate new usage: 0 requests, 0 cache hits, 0 parse errors, 0 tokens, $0 reported spend; no response usage fields were received.',
    '- Pilot, Score/Noul metrics, thresholds, group bootstrap, score plots, scatter plot, review burden, and missed-case count were not measured. The exact question design and predeclared rule are saved separately.','',
    '## Answers and limits','',
    '1. Whether Noul separates abnormal evidence better than v4 Score alone: not determined; no new paired responses.',
    '2. Number of near_failure samples missed by the Noul gate: not measured.',
    '3. Practical value after review burden: not measured.',
    '4. Treat as an exploratory candidate; operational adoption is deferred. TTF is not a measured current-state label.',
    '5. Needed next: unused source_file::sequence_index groups across all nine cells for separate development, grouped CV, and untouched evaluation; measured current equipment-state/fault labels aligned to each window; and prospective or external groups with timestamps and maintenance/fault adjudication.'
]
(OUT_DIR / 'run_summary.md').write_text('\n'.join(summary_lines)+'\n',encoding='utf-8')
print('\nREQUEST DESIGN AND GATE')
print('Two questions in one request would use the exact saved v4 Score and the specified Noul question.')
print('Noul values <0.4 pass low; 0.4-0.6 inclusive require review; >0.6 pass to v4 thresholds. Missing => review.')
print('New budget: 300 requests / $0.05; planned upper design: 270 requests including pilot.')
print('New usage: requests 0, cache hits 0, parse errors 0, tokens 0/0, reported spend $0.00; no API usage fields received.')
print('Pilot/development/CV/evaluation/plots skipped because feasibility is false.')
print('Conclusion: no comparison estimate; exploratory candidate only and operational adoption deferred.')


## Noul exploratory paired run on observed 05b groups

The unused fixed-validation groups cannot fill `leak / near_failure`, so this fallback deliberately reuses the saved 05b sample groups. It asks exact v4 Score and the specified Noul question together for each window. Development and the previously observed 05b holdout remain separate by `source_file::sequence_index`, but the holdout is an **observed exploratory cohort**, not an independent holdout. Its earlier Score and TTF results have already been viewed.

Only the new `artifacts/jev_prompt_optimization/noul_followup/reused_groups_exploratory/` directory receives requests, cache, settings and results. The pilot uses five development windows per target-by-TTF cell. Model, TypeSafe provider, both answer parsers, usage fields and the projected 270-request comparison must pass before continuing. The new hard caps are 300 actual requests and $0.05 OpenRouter-reported cost. Neither Noul nor Score is a measured failure probability. The numeric Score thresholds will be selected within development groups only, with fold training groups used for CV.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / '.venv' / 'Lib' / 'site-packages'))
sys.path.insert(0, str(ROOT / 'src'))
from fabjudge.jev_noul_followup import run_pilot
pilot_summary = run_pilot(ROOT)
print('Pilot status:', pilot_summary['status'])


## Noul development analysis and grouped cross-validation

This exploratory reuse of the previously observed 05b groups selects thresholds only on development rows. Each cross-validation fold refits thresholds on its own training groups. The original 05b holdout was previously observed and cannot provide independent validation.


In [ ]:
from fabjudge.jev_noul_followup import run_development
development_result = run_development(ROOT)


## Locked rule on previously observed 05b evaluation groups

Apply the development-locked thresholds once to the 90 saved 05b holdout windows. These groups are disjoint from this follow-up development split but their prior 05b outcomes were already observed; this is an exploratory paired comparison, not an independent holdout. Group-paired bootstrap and plots use the same fresh Score and Noul responses.


In [ ]:
from fabjudge.jev_noul_evaluation import run_evaluation
evaluation_result = run_evaluation(ROOT)


# Noul 후속 실험: 실행 결과 (탐색적 그룹 재사용)

**평가 지위:** 이번 90개 평가 window의 그룹은 이번 개발 그룹과 겹치지 않지만, 이전 05b에서 이미 관찰된 그룹이다. 따라서 독립 holdout 검증이 아니다. TTF 구간은 평가용 대리 라벨이며 현재 설비 비정상 상태의 정답이 아니다.

## 기존 05a·스케일·05b 결과 (새 실험과 분리)

| 항목 | 이전에 확인한 사실 |
| --- | --- |
| 분석 단위 | 길이 100, 간격 50인 겹치는 window; 그룹은 source_file::sequence_index |
| 입력 | 기존 20개 특성, train-only Robust, 1× |
| 스케일 | 0.5×·2×는 기존 1×보다 종합적으로 낫지 않았고, 0.1×·10×는 같은 실행 배치의 fresh 1×보다 낫지 않았다. 과거 1×와 fresh 1×의 차이는 스케일 효과로 해석하지 않는다. |
| 05b 선택 | v5가 개발 선택 규칙에서 탈락해 v4 유지; 기존 3단계 임계값은 탐색 후보, 운영 채택 보류 |
| 05b 사용량 | 552회 상한 소진; 기존 응답·캐시·설정·선택 기록·diagnostics는 읽기만 함 |

기존 결과의 원본 파일과 최초 표본 감사는 상위 `noul_followup/`의 preflight 산출물에 보존했다.

## 새 실험 설계와 표본 감사

05a·05b에서 쓰지 않은 그룹만으로 9개 셀을 구성할 수 없었다. 특히 validation의 leak/고장근접 가용 그룹 8개(05b 개발 후보 6, 평가 후보 2)가 모두 05b에서 사용됐다. 원래 독립 표본 계획의 균형 셀 용량은 0이었다. 사용자 후속 지시에 따라 05b의 270개 window를 **탐색적으로 재사용**했다.

**API 호출 전 가용성 감사:** 그룹 미사용 수치는 해당 셀에서 05b가 이미 사용한 그룹을 뺀 상한이며, 다른 셀에서 사용된 그룹과의 교차를 고려하면 더 줄 수 있다.

| 타깃 / TTF 구간 | 기존 개발 window / 그룹 | 기존 평가 window / 그룹 | 05b 미사용 개발 / 평가 그룹 상한 |
| --- | --- | --- | --- |
| leak / 정상 | 59245 / 2463 | 17833 / 776 | 2443 / 766 |
| leak / 경계 | 753 / 27 | 121 / 6 | 15 / 2 |
| leak / 고장근접 | 163 / 6 | 57 / 2 | 0 / 0 |
| pressure_high / 정상 | 97855 / 5071 | 32266 / 1676 | 5051 / 1666 |
| pressure_high / 경계 | 1500 / 76 | 495 / 24 | 58 / 17 |
| pressure_high / 고장근접 | 247 / 17 | 42 / 5 | 6 / 2 |
| pressure_low / 정상 | 119091 / 5475 | 38982 / 1816 | 5455 / 1806 |
| pressure_low / 경계 | 3112 / 165 | 1279 / 66 | 146 / 56 |
| pressure_low / 고장근접 | 556 / 33 | 197 / 15 | 22 / 9 |

[전체 가용성 감사](../artifacts/jev_prompt_optimization/noul_followup/feasibility_audit.csv)

| 구간 | 개발 | 과거 관찰 평가 |
| --- | --- | --- |
| leak / 정상 | 20 | 10 |
| leak / 경계 | 20 | 10 |
| leak / 고장근접 | 20 | 10 |
| pressure_high / 정상 | 20 | 10 |
| pressure_high / 경계 | 20 | 10 |
| pressure_high / 고장근접 | 20 | 10 |
| pressure_low / 정상 | 20 | 10 |
| pressure_low / 경계 | 20 | 10 |
| pressure_low / 고장근접 | 20 | 10 |

개발 180개/123그룹, 과거 관찰 평가 90개/57그룹이다. 두 부분 사이 그룹 중복과 동일 물리 window 중복은 0이었다. 그룹을 통째로 분리해 같은 run의 겹치는 인접 window가 양쪽에 들어가지 않도록 했다. 표본 복제나 부족 셀 임의 보충은 하지 않았다.

요청 모델은 `typesafe/jev-1.13`, API는 `https://openrouter.ai/api/alpha/decisions`로 고정했다. 각 요청 본문에는 모델, 20개 Robust 1× 숫자 특성, **정확히 재사용한 v4 Score 질문과 새 Noul 질문**만 넣었다. Noul은 현재 비정상 근거의 예/아니오 질문이며 true/false 문구는 [질문 정의](../artifacts/jev_prompt_optimization/noul_followup/reused_groups_exploratory/questions.json)에 있다. Noul 0.5 부근은 판단이 갈린 상태로 취급하며 TTF 경계·Score 중간값·Jev `Uncertain`과 동일시하지 않는다. 두 출력은 실제 고장 확률이 아니다. [요청 본문 감사](../artifacts/jev_prompt_optimization/noul_followup/reused_groups_exploratory/request_body_audit.json)에서 금지 메타데이터와 원시 특성 저장 여부를 확인할 수 있다.

고정한 규칙: `Noul < 0.4 → 낮은 위험`, `0.4 ≤ Noul ≤ 0.6 → 검토 필요`, `Noul > 0.6 → 같은 요청의 v4 Score 임계값으로 3단계 분류`. Score 결측이면 검토 필요로 처리한다. Score는 `≤t1` 낮음, `(t1,t2]` 중간, `>t2` 높음이다. 연속 임계값은 개발 데이터에서 타깃 평균 balanced accuracy, 동률이면 macro-F1, 다시 동률이면 작은 임계값 순으로 선택했다. 이 규칙과 결측·동점 처리는 개발 결과를 보기 전에 [분석 계획](../artifacts/jev_prompt_optimization/noul_followup/reused_groups_exploratory/analysis_plan_locked_before_development.json)에 저장했다.

## 파일럿과 새 사용량

| 항목 | 45개 파일럿 | 새 실험 전체 | 이전 05b (별도) |
| --- | --- | --- | --- |
| 요청 | 45 | 270 | 552 |
| 성공 응답 | 45 | 270 | 552 |
| 캐시 적중 | 0 | 개발 재실행에서 45 (파일럿 응답 재사용) | 90 |
| 파싱 오류 | 0 | 0 | 0 |
| 입력 토큰 | 47560 | 286068 | 559734 |
| 출력 토큰 | 1530 | 9180 | 9936 |
| 보고 비용 | $0.00199752 | $0.01201486 | $0.02350883 |

새 실험 hard cap은 300회/$0.05이며, 270회/$0.012014856으로 종료했다. 새 응답 모델은 `typesafe/jev-1.13-20260917` 270회, 제공자는 `TypeSafe` 270회였다. 키와 원시 PHM 특성 벡터는 산출물에 저장하지 않았다. 파일럿은 9개 개발 셀에서 각 5개로 구성됐고 모델·제공자·파싱·세 usage 필드를 확인한 뒤 진행했다.

## 개발: 연속값과 Noul 예/아니오 판단

| 타깃 | v4 ROC-AUC | Noul ROC-AUC | v4 PR-AUC* | Noul PR-AUC* | Noul BA @0.5 | Noul 고장근접 recall @0.5 |
| --- | --- | --- | --- | --- | --- | --- |
| leak | 0.398 | 0.411 | 0.459 | 0.477 | 0.500 | 5.0% |
| pressure_high | 0.617 | 0.576 | 0.545 | 0.520 | 0.475 | 0.0% |
| pressure_low | 0.721 | 0.619 | 0.683 | 0.574 | 0.500 | 5.0% |
| macro | 0.579 | 0.535 | 0.563 | 0.523 | 0.492 | 3.3% |

정상과 고장근접 TTF 대리 구간만 사용한 이진 대조다. 경계 구간은 여기서 제외했고 아래 3단계 평가에는 포함했다. *PR-AUC와 precision은 균형 추출 표본의 값이며 실제 운전 발생률로 해석할 수 없다. 실제 현재 비정상 상태 라벨은 없다.

개발 전체에서 Noul은 0.28–0.55, 평가 전체에서는 0.26–0.51이었다. 평가에서 `Noul ≥ 0.5`는 4/90개, `Noul > 0.6`은 0개였다. 같은 응답의 Score와 Noul의 평가 Pearson 상관은 0.921이었다. 관문 뒤 Score가 추가 위험 단계를 자동 판정할 사례가 없었다.

## 개발, 그룹 보존 교차검증, 평가의 짝지은 비교

개발에서 선택한 Score 임계값(0–1 정규화)은 v4 단독 `t1=0.4525, t2=0.5975`, Noul 관문 뒤 Score `t1=0, t2=0.2925`였다. 3-fold 그룹 보존 교차검증은 매 fold에서 학습 그룹만으로 양쪽 임계값을 다시 선택했으며, 평가 90개에는 개발에서 잠근 값을 한 번 적용했다. 평가 그룹은 이전 05b에 노출됐으므로 아래 값은 탐색적이다.

| 표본 | 방식 | macro BA | macro-F1 | 정상→높음 | 고장근접→낮음 | 검토 필요 | 자동 판정 | 자동 표본 오류율† |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 개발 적합 | v4 단독 | 0.456 | 0.434 | 23.3% | 18.3% | 0.0% | 100.0% | 54.4% |
| 개발 적합 | Noul 관문+v4 | 0.183 | 0.142 | 0.0% | 53.3% | 46.7% | 53.3% | 64.7% |
| 개발 그룹 CV OOF | v4 단독 | 0.428 | 0.417 | 26.7% | 18.3% | 0.0% | 100.0% | 57.2% |
| 개발 그룹 CV OOF | Noul 관문+v4 | 0.183 | 0.142 | 0.0% | 53.3% | 46.7% | 53.3% | 64.7% |
| 과거 관찰 평가 | v4 단독 | 0.422 | 0.410 | 30.0% | 16.7% | 0.0% | 100.0% | 57.8% |
| 과거 관찰 평가 | Noul 관문+v4 | 0.200 | 0.160 | 0.0% | 40.0% | 52.2% | 47.8% | 59.7% |

†표에는 타깃별 자동 오류율의 평균을 사용했다. 과거 관찰 평가의 실제 자동 판정 43개 중 오류는 25개(58.1%)다. 검토 필요는 별도 결과이며 balanced accuracy/recall/F1에서는 정답이 아닌 것으로 계산했다.

### 과거 관찰 평가: 타깃별 결과

| 타깃 | 방식 | BA | macro-F1 | 정상→높음 | 고장근접→낮음 | 검토 필요 | 자동 오류율 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| leak | v4 단독 | 0.333 | 0.337 | 20.0% | 20.0% | 0.0% | 66.7% |
| leak | Noul 관문+v4 | 0.233 | 0.187 | 0.0% | 60.0% | 50.0% | 53.3% |
| pressure_high | v4 단독 | 0.500 | 0.479 | 20.0% | 0.0% | 0.0% | 50.0% |
| pressure_high | Noul 관문+v4 | 0.267 | 0.198 | 0.0% | 20.0% | 43.3% | 52.9% |
| pressure_low | v4 단독 | 0.433 | 0.413 | 50.0% | 30.0% | 0.0% | 56.7% |
| pressure_low | Noul 관문+v4 | 0.100 | 0.095 | 0.0% | 40.0% | 63.3% | 72.7% |
| macro | v4 단독 | 0.422 | 0.410 | 30.0% | 16.7% | 0.0% | 57.8% |
| macro | Noul 관문+v4 | 0.200 | 0.160 | 0.0% | 40.0% | 52.2% | 59.7% |


### 과거 관찰 평가: confusion matrix

| 타깃 | 방식 | 대리 라벨 | 판정 낮음 | 판정 중간 | 판정 높음 | 검토 필요 |
| --- | --- | --- | --- | --- | --- | --- |
| leak | v4 단독 | 정상 | 3 | 5 | 2 | 0 |
| leak | v4 단독 | 경계 | 1 | 3 | 6 | 0 |
| leak | v4 단독 | 고장근접 | 2 | 4 | 4 | 0 |
| leak | Noul 관문+v4 | 정상 | 7 | 0 | 0 | 3 |
| leak | Noul 관문+v4 | 경계 | 2 | 0 | 0 | 8 |
| leak | Noul 관문+v4 | 고장근접 | 6 | 0 | 0 | 4 |
| pressure_high | v4 단독 | 정상 | 2 | 6 | 2 | 0 |
| pressure_high | v4 단독 | 경계 | 2 | 7 | 1 | 0 |
| pressure_high | v4 단독 | 고장근접 | 0 | 4 | 6 | 0 |
| pressure_high | Noul 관문+v4 | 정상 | 8 | 0 | 0 | 2 |
| pressure_high | Noul 관문+v4 | 경계 | 7 | 0 | 0 | 3 |
| pressure_high | Noul 관문+v4 | 고장근접 | 2 | 0 | 0 | 8 |
| pressure_low | v4 단독 | 정상 | 2 | 3 | 5 | 0 |
| pressure_low | v4 단독 | 경계 | 0 | 7 | 3 | 0 |
| pressure_low | v4 단독 | 고장근접 | 3 | 3 | 4 | 0 |
| pressure_low | Noul 관문+v4 | 정상 | 3 | 0 | 0 | 7 |
| pressure_low | Noul 관문+v4 | 경계 | 4 | 0 | 0 | 6 |
| pressure_low | Noul 관문+v4 | 고장근접 | 4 | 0 | 0 | 6 |


### 과거 관찰 평가: 클래스별 precision·recall

| 타깃 | 방식 | 클래스 | 개수 | precision* | recall |
| --- | --- | --- | --- | --- | --- |
| leak | v4 단독 | 정상 | 10 | 0.500 | 0.300 |
| leak | v4 단독 | 경계 | 10 | 0.250 | 0.300 |
| leak | v4 단독 | 고장근접 | 10 | 0.333 | 0.400 |
| leak | Noul 관문+v4 | 정상 | 10 | 0.467 | 0.700 |
| leak | Noul 관문+v4 | 경계 | 10 | 0.000 | 0.000 |
| leak | Noul 관문+v4 | 고장근접 | 10 | 0.000 | 0.000 |
| pressure_high | v4 단독 | 정상 | 10 | 0.500 | 0.200 |
| pressure_high | v4 단독 | 경계 | 10 | 0.412 | 0.700 |
| pressure_high | v4 단독 | 고장근접 | 10 | 0.667 | 0.600 |
| pressure_high | Noul 관문+v4 | 정상 | 10 | 0.471 | 0.800 |
| pressure_high | Noul 관문+v4 | 경계 | 10 | 0.000 | 0.000 |
| pressure_high | Noul 관문+v4 | 고장근접 | 10 | 0.000 | 0.000 |
| pressure_low | v4 단독 | 정상 | 10 | 0.400 | 0.200 |
| pressure_low | v4 단독 | 경계 | 10 | 0.538 | 0.700 |
| pressure_low | v4 단독 | 고장근접 | 10 | 0.333 | 0.400 |
| pressure_low | Noul 관문+v4 | 정상 | 10 | 0.273 | 0.300 |
| pressure_low | Noul 관문+v4 | 경계 | 10 | 0.000 | 0.000 |
| pressure_low | Noul 관문+v4 | 고장근접 | 10 | 0.000 | 0.000 |
| macro | v4 단독 | 정상 | 30 | 0.467 | 0.233 |
| macro | v4 단독 | 경계 | 30 | 0.400 | 0.567 |
| macro | v4 단독 | 고장근접 | 30 | 0.444 | 0.467 |
| macro | Noul 관문+v4 | 정상 | 30 | 0.403 | 0.600 |
| macro | Noul 관문+v4 | 경계 | 30 | 0.000 | 0.000 |
| macro | Noul 관문+v4 | 고장근접 | 30 | 0.000 | 0.000 |

*precision은 균형 표본에서의 값이다. 판정된 클래스가 없을 때 precision은 0으로 정의했다. 자세한 개발·평가 CSV는 [개발 지표](../artifacts/jev_prompt_optimization/noul_followup/reused_groups_exploratory/development_metrics.csv), [평가 지표](../artifacts/jev_prompt_optimization/noul_followup/reused_groups_exploratory/observed_05b_holdout_metrics.csv), [평가 confusion](../artifacts/jev_prompt_optimization/noul_followup/reused_groups_exploratory/observed_05b_holdout_confusion.csv)에 있다.

## 그룹 단위 paired bootstrap과 민감도

| 과거 관찰 평가 지표 | 원자료 점 추정 | 그룹 paired 95% 구간 | 유효 재표집 |
| --- | --- | --- | --- |
| macro BA 차이 | -0.222 | [-0.366, -0.091] | 814/1000 |
| macro-F1 차이 | -0.250 | [-0.344, -0.110] | 814/1000 |
| 고장근접→낮음 차이 | 0.233 | [0.036, 0.454] | 827/1000 |
| 검토 필요 비율 | 0.522 | [0.340, 0.676] | 1000/1000 |
| 자동 표본 오류율 | 0.597 | [0.358, 0.745] | 999/1000 |
| 관문에서 놓친 고장근접 비율 | 0.400 | [0.061, 0.663] | 827/1000 |

같은 `source_file::sequence_index` 그룹의 모든 표본과 두 방식의 결과를 함께 재표집했다. 점 추정치는 재표집 평균이 아니라 원자료에서 계산했다. 일부 재표집에서 특정 타깃/클래스가 사라져 BA의 유효 반복은 814/1000회였으며, 이는 적은 그룹에 따른 불안정성을 드러낸다. 개발 CV OOF의 BA 차이는 -0.244 (95% 구간 [-0.322, -0.173])였다.
평가에서 그룹 하나씩 제외한 BA 차이 범위는 -0.289–-0.178, 검토 필요 비율 범위는 48.1%–56.4%였다. [그룹별 민감도 전체](../artifacts/jev_prompt_optimization/noul_followup/reused_groups_exploratory/observed_05b_holdout_group_sensitivity.csv)를 보라.

## Noul 관문이 놓친 고장근접 표본

개발에서는 **32/60 (53.3%)**, 과거 관찰 평가에서는 **12/30 (40.0%)** 고장근접 대리 표본이 `Noul < 0.4`라서 낮은 위험으로 자동 판정됐다. 평가 타깃별로 leak **6/10**, pressure_high **2/10**, pressure_low **4/10**이었다. 이 12개에 대한 Score 단독 판정은 중간 7개, 낮음 5개였다. [해당 사례 목록](../artifacts/jev_prompt_optimization/noul_followup/reused_groups_exploratory/observed_05b_holdout_gate_missed_near_failure.csv)에 원시 특성 없이 식별값, TTF 대리 구간, Score/Noul 판정만 저장했다.

![Noul–Score 산점도](../artifacts/jev_prompt_optimization/noul_followup/reused_groups_exploratory/noul_score_scatter.png)

![점수 분포](../artifacts/jev_prompt_optimization/noul_followup/reused_groups_exploratory/score_noul_distributions.png)

## 질문에 대한 결론

1. **Noul이 더 잘 구분했는가?** 현재 비정상 상태의 참 라벨이 없어 직접 판정할 수 없다. 정상 대 고장근접 TTF 대리 구간에서는 개발 macro ROC-AUC가 Noul 0.535, 같은 요청의 v4 Score 0.579로 Noul의 우위가 보이지 않았다.
2. **관문에서 놓친 고장근접 표본은?** 평가 12/30 (40.0%), 개발 32/60 (53.3%)였다. 평가 12개 중 Score 단독은 7개를 중간으로 구분했다.
3. **검토 필요 비율을 감안한 단계 구분 이득은?** 보이지 않았다. 평가 macro BA는 0.422→0.200, macro-F1은 0.410→0.160으로 낮아졌고, 47/90 (52.2%)가 검토 필요였다. 자동 판정 43개는 모두 낮은 위험이었으며 25개가 대리 라벨과 달랐다. 정상의 높은 위험 판정은 9/30→0/30으로 줄었지만, 고장근접의 낮은 위험 판정은 5/30→12/30으로 늘었다.
4. **채택 판단은?** 그룹 재사용, 빈약한 leak/고장근접 그룹, 814/1000회만 유효한 평가 bootstrap, 현재 상태 정답 부재를 감안해 탐색 후보로만 남기고 운영 채택을 보류한다. 이 결과를 독립 검증으로 제시하지 않는다.
5. **운영 판단에 필요한 자료는?** 9개 타깃×TTF 셀, 특히 leak/고장근접을 채울 05a·05b 미사용 `source_file::sequence_index` 그룹과 별도 개발·평가 그룹; window 시점에 정렬된 실제 설비 상태·고장·정비 판정 라벨; 날짜·run 경계가 확인되는 전향적 또는 외부 검증 자료와 현장 검토 기록이 필요하다.

Noul 0.5는 예/아니오 판단이 갈린 지점일 뿐 중간 심각도가 아니다. Score와 Noul은 실제 고장 확률이 아니다. 균형 표본의 precision/PR-AUC를 운영 발생률에 적용하지 않는다.
